In [1]:
import pandas as pd

df_billboard_hot100_songs_final = pd.read_csv('/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/billboard_hot100_songs_final.csv')
df_billboard_200_albums_final = pd.read_csv('/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/billboard_200_albums_final.csv')
df_artists = pd.read_csv('df_artists.csv')


In [2]:
# This code creates df_songs from df_billboard_hot100_songs_final by:
# 1. Renaming performer_normalized → name, performer → performer_pre_normalized,
#    original_performer → original_performer_name, and
#    original_performer_normalized → original_performer_name_normalized
# 2. Reordering columns so that 'name' appears immediately after 'title',
#    and the original_performer columns follow performer_pre_normalized in sequence.

df_songs = df_billboard_hot100_songs_final.rename(columns={
    'performer_normalized': 'name',
    'performer': 'performer_pre_normalized',
    'original_performer': 'original_performer_name',
    'original_performer_normalized': 'original_performer_name_normalized',
})

# Build new column order: insert 'name' after 'title', then fix original_performer block
cols = list(df_songs.columns)

# Remove the columns we want to reposition
for col in ['name', 'original_performer_name', 'original_performer_name_normalized']:
    cols.remove(col)

# Insert 'name' after 'title'
title_idx = cols.index('title')
cols.insert(title_idx + 1, 'name')

# Insert original_performer columns after 'performer_pre_normalized'
perf_idx = cols.index('performer_pre_normalized')
cols.insert(perf_idx + 1, 'original_performer_name')
cols.insert(perf_idx + 2, 'original_performer_name_normalized')

df_songs = df_songs[cols]
df_songs.head()


,title,name,performer_pre_normalized,original_performer_name,original_performer_name_normalized,peak_pos,wks_on_chart,first_charting_year
0,It's All In The Game,tommy edwards,Tommy Edwards,NaN,NaN,1,22,1958
1,It's Only Make Believe,conway twitty,Conway Twitty,NaN,NaN,1,21,1958
2,Little Star,the elegants,The Elegants,NaN,NaN,1,17,1958
3,Nel Blu Dipinto Di Blu (Volaré),domenico modugno,Domenico Modugno,NaN,NaN,1,16,1958
4,Poor Little Fool,ricky nelson,Ricky Nelson,NaN,NaN,1,11,1958


In [7]:
# This code adds musicbrainz_recording_mbid to df_songs by querying the local
# MusicBrainz PostgreSQL database. Rather than matching on artist name (which
# fails due to normalization differences), it uses the already-verified
# musicbrainz_artist_id UUID to join against artist.gid, which is exact and
# reliable. Runs in a background thread so the stop button works cleanly.

import psycopg2
import psycopg2.extras
import threading

DB_PARAMS = {
    'dbname':  'musicbrainz_db',
    'user':    'musicbrainz',
    'password':'musicbrainz',
    'host':    'localhost',
    'port':    5432
}

# Use (title, artist_mbid) pairs instead of (title, artist_name)
pairs = (
    df_songs[df_songs['musicbrainz_artist_id'].notna()][['title', 'musicbrainz_artist_id']]
    .drop_duplicates()
    .dropna()
    .values.tolist()
)
print(f"Unique (title, artist_mbid) pairs to look up: {len(pairs):,}")

results = []
query_error = []

def run_query():
    conn = psycopg2.connect(**DB_PARAMS)
    try:
        with conn.cursor() as cur:
            cur.execute("SET jit = off")
            cur.execute("SET statement_timeout = '1800s'")

            cur.execute("""
                CREATE TEMP TABLE input_pairs (title text, artist_mbid uuid)
                ON COMMIT DROP
            """)

            psycopg2.extras.execute_values(
                cur,
                "INSERT INTO input_pairs VALUES %s",
                pairs,
                page_size=1000
            )

            cur.execute("""
                SELECT DISTINCT ON (ip.title, ip.artist_mbid)
                    ip.title,
                    ip.artist_mbid::text,
                    r.gid::text AS recording_mbid
                FROM input_pairs ip
                JOIN recording r ON r.name = ip.title
                JOIN artist_credit_name acn ON acn.artist_credit = r.artist_credit
                JOIN artist a ON a.id = acn.artist AND a.gid = ip.artist_mbid
                ORDER BY ip.title, ip.artist_mbid
            """)

            results.extend(cur.fetchall())
        conn.commit()
    except Exception as e:
        query_error.append(e)
    finally:
        conn.close()

thread = threading.Thread(target=run_query, daemon=True)
thread.start()
thread.join()

if query_error:
    raise query_error[0]

pair_to_recording_mbid = {(title, artist_mbid): rec_mbid for title, artist_mbid, rec_mbid in results}

df_songs['musicbrainz_recording_mbid'] = df_songs.apply(
    lambda r: pair_to_recording_mbid.get((r['title'], r['musicbrainz_artist_id'])), axis=1
)

matched_recs = df_songs['musicbrainz_recording_mbid'].notna().sum()
print(f"Recording MBIDs matched: {matched_recs:,}/{len(df_songs):,} ({matched_recs/len(df_songs)*100:.1f}%)")

df_songs.head()


Unique (title, artist_mbid) pairs to look up: 35,093
Recording MBIDs matched: 23,343/38,383 (60.8%)


,title,name,performer_pre_normalized,original_performer_name,original_performer_name_normalized,peak_pos,wks_on_chart,first_charting_year,musicbrainz_artist_id,musicbrainz_recording_mbid
0,It's All In The Game,tommy edwards,Tommy Edwards,NaN,NaN,1,22,1958,0b1a0ca5-52cb-4d1d-8344-746f905ae115,7cf06b8f-a898-4f2c-a589-a98af8af6001
1,It's Only Make Believe,conway twitty,Conway Twitty,NaN,NaN,1,21,1958,a3c60d26-90d6-4788-ba9b-a89693fc396d,4123e40f-d667-4634-919d-6e1d77a19934
2,Little Star,the elegants,The Elegants,NaN,NaN,1,17,1958,91919ad2-80cb-4077-bbb3-2803fa129fab,e15d4722-47a7-4874-99f6-285529503569
3,Nel Blu Dipinto Di Blu (Volaré),domenico modugno,Domenico Modugno,NaN,NaN,1,16,1958,b8e60837-e646-4f18-8f92-27d1173511b0,None
4,Poor Little Fool,ricky nelson,Ricky Nelson,NaN,NaN,1,11,1958,28d0c272-4d51-4c24-b31f-e20aac2ba7de,100b7f30-1519-427a-adc2-2b0d6a737db1


In [9]:
# This cell does a second-pass lookup for songs that didn't get a recording MBID
# in the first pass. It uses case-insensitive title matching (LOWER) on the
# unmatched songs only. Since we're already filtering by artist UUID, the result
# set is much smaller and faster than a full case-insensitive scan.
# Runs in a background thread so the stop button works cleanly.

import psycopg2
import psycopg2.extras
import threading

DB_PARAMS = {
    'dbname':  'musicbrainz_db',
    'user':    'musicbrainz',
    'password':'musicbrainz',
    'host':    'localhost',
    'port':    5432
}

unmatched_pairs = (
    df_songs[
        df_songs['musicbrainz_recording_mbid'].isna() &
        df_songs['musicbrainz_artist_id'].notna()
    ][['title', 'musicbrainz_artist_id']]
    .drop_duplicates()
    .dropna()
    .values.tolist()
)
print(f"Unmatched pairs to retry with case-insensitive matching: {len(unmatched_pairs):,}")

results = []
query_error = []

def run_query():
    conn = psycopg2.connect(**DB_PARAMS)
    try:
        with conn.cursor() as cur:
            cur.execute("SET jit = off")
            cur.execute("SET statement_timeout = '1800s'")

            cur.execute("""
                CREATE TEMP TABLE input_pairs (title text, artist_mbid uuid)
                ON COMMIT DROP
            """)

            psycopg2.extras.execute_values(
                cur,
                "INSERT INTO input_pairs VALUES %s",
                unmatched_pairs,
                page_size=1000
            )

            cur.execute("""
                SELECT DISTINCT ON (ip.title, ip.artist_mbid)
                    ip.title,
                    ip.artist_mbid::text,
                    r.gid::text AS recording_mbid
                FROM input_pairs ip
                JOIN recording r ON LOWER(r.name) = LOWER(ip.title)
                JOIN artist_credit_name acn ON acn.artist_credit = r.artist_credit
                JOIN artist a ON a.id = acn.artist AND a.gid = ip.artist_mbid
                ORDER BY ip.title, ip.artist_mbid
            """)

            results.extend(cur.fetchall())
        conn.commit()
    except Exception as e:
        query_error.append(e)
    finally:
        conn.close()

thread = threading.Thread(target=run_query, daemon=True)
thread.start()
thread.join()

if query_error:
    raise query_error[0]

pair_to_recording_mbid = {(title, artist_mbid): rec_mbid for title, artist_mbid, rec_mbid in results}

# Fill in only the previously unmatched rows
mask = df_songs['musicbrainz_recording_mbid'].isna() & df_songs['musicbrainz_artist_id'].notna()
df_songs.loc[mask, 'musicbrainz_recording_mbid'] = df_songs[mask].apply(
    lambda r: pair_to_recording_mbid.get((r['title'], r['musicbrainz_artist_id'])), axis=1
)

matched_recs = df_songs['musicbrainz_recording_mbid'].notna().sum()
print(f"Recording MBIDs matched: {matched_recs:,}/{len(df_songs):,} ({matched_recs/len(df_songs)*100:.1f}%)")

df_songs.head()


Unmatched pairs to retry with case-insensitive matching: 11,807
Recording MBIDs matched: 28,585/38,383 (74.5%)


,title,name,performer_pre_normalized,original_performer_name,original_performer_name_normalized,peak_pos,wks_on_chart,first_charting_year,musicbrainz_artist_id,musicbrainz_recording_mbid
0,It's All In The Game,tommy edwards,Tommy Edwards,NaN,NaN,1,22,1958,0b1a0ca5-52cb-4d1d-8344-746f905ae115,7cf06b8f-a898-4f2c-a589-a98af8af6001
1,It's Only Make Believe,conway twitty,Conway Twitty,NaN,NaN,1,21,1958,a3c60d26-90d6-4788-ba9b-a89693fc396d,4123e40f-d667-4634-919d-6e1d77a19934
2,Little Star,the elegants,The Elegants,NaN,NaN,1,17,1958,91919ad2-80cb-4077-bbb3-2803fa129fab,e15d4722-47a7-4874-99f6-285529503569
3,Nel Blu Dipinto Di Blu (Volaré),domenico modugno,Domenico Modugno,NaN,NaN,1,16,1958,b8e60837-e646-4f18-8f92-27d1173511b0,None
4,Poor Little Fool,ricky nelson,Ricky Nelson,NaN,NaN,1,11,1958,28d0c272-4d51-4c24-b31f-e20aac2ba7de,100b7f30-1519-427a-adc2-2b0d6a737db1


In [14]:
# This cell shows a random sample and year distribution of songs still unmatched
# after the third pass, to identify any remaining patterns worth fixing.

still_unmatched = df_songs[
    df_songs['musicbrainz_recording_mbid'].isna() &
    df_songs['musicbrainz_artist_id'].notna()
]
print(f"Still unmatched: {len(still_unmatched):,}")
print("\nRandom sample:")
print(still_unmatched[['title', 'name']].drop_duplicates().sample(30, random_state=42))
print("\nYear distribution of unmatched songs:")
print(still_unmatched['first_charting_year'].value_counts().sort_index())


Still unmatched: 6,127

Random sample:
                                              title  \
6603                                Love Me Forever   
11519                                    Only Women   
12425                                  Close To You   
13277                              I Don't Wanna Go   
19415                           I'll Take You There   
13122        I'm Not Gonna Let It Bother Me Tonight   
5984                   Can I Get To Know You Better   
4546                                      Big Party   
5867   Don't Worry Mother, Your Son's Heart Is Pure   
6317                Do It Again A Little Bit Slower   
6878                            Mercy, Mercy, Mercy   
8699                                     Sweetheart   
28402                                      My Hitta   
32824                                  Whats Poppin   
11411           He Don't Love You (Like I Love You)   
2932                                     Jellybread   
7587                Listen

In [16]:
# This code adds two label columns to df_songs by querying the local MusicBrainz
# PostgreSQL database. For each song with a recording MBID, it finds the earliest
# release that contains that recording, then returns all labels associated with
# that release (a single release can have multiple labels, e.g. one for the
# imprint and one for distribution). Songs without a recording MBID are skipped.
# Results are stored as lists in:
#   - label: list of label names from the earliest release
#   - label_mbid: list of corresponding label MusicBrainz UUIDs
# Runs in a background thread so the stop button works cleanly.

import psycopg2
import psycopg2.extras
import threading

DB_PARAMS = {
    'dbname':  'musicbrainz_db',
    'user':    'musicbrainz',
    'password':'musicbrainz',
    'host':    'localhost',
    'port':    5432
}

recording_mbids = (
    df_songs['musicbrainz_recording_mbid']
    .dropna()
    .unique()
    .tolist()
)
print(f"Unique recording MBIDs to look up: {len(recording_mbids):,}")

results = []
query_error = []

def run_query():
    conn = psycopg2.connect(**DB_PARAMS)
    try:
        with conn.cursor() as cur:
            cur.execute("SET jit = off")
            cur.execute("SET statement_timeout = '1800s'")

            cur.execute("""
                CREATE TEMP TABLE input_recordings (recording_mbid uuid)
                ON COMMIT DROP
            """)

            psycopg2.extras.execute_values(
                cur,
                "INSERT INTO input_recordings VALUES %s",
                [(mbid,) for mbid in recording_mbids],
                page_size=1000
            )

            cur.execute("""
                WITH earliest_release AS (
                    -- For each recording, find the earliest release it appears on
                    -- using release_event which combines release_country and
                    -- release_unknown_country
                    SELECT DISTINCT ON (r.gid)
                        r.gid AS recording_mbid,
                        rel.id AS release_id
                    FROM input_recordings ir
                    JOIN recording r ON r.gid = ir.recording_mbid
                    JOIN track t ON t.recording = r.id
                    JOIN medium m ON m.id = t.medium
                    JOIN release rel ON rel.id = m.release
                    LEFT JOIN release_event re ON re.release = rel.id
                    ORDER BY
                        r.gid,
                        re.date_year  NULLS LAST,
                        re.date_month NULLS LAST,
                        re.date_day   NULLS LAST
                )
                SELECT
                    er.recording_mbid::text,
                    array_agg(l.name      ORDER BY rl.id) AS label_names,
                    array_agg(l.gid::text ORDER BY rl.id) AS label_mbids
                FROM earliest_release er
                JOIN release_label rl ON rl.release = er.release_id
                JOIN label l ON l.id = rl.label
                GROUP BY er.recording_mbid
            """)

            results.extend(cur.fetchall())
        conn.commit()
    except Exception as e:
        query_error.append(e)
    finally:
        conn.close()

thread = threading.Thread(target=run_query, daemon=True)
thread.start()
thread.join()

if query_error:
    raise query_error[0]

recording_to_labels = {
    recording_mbid: (label_names, label_mbids)
    for recording_mbid, label_names, label_mbids in results
}

df_songs['label'] = df_songs['musicbrainz_recording_mbid'].map(
    lambda x: recording_to_labels.get(x, (None, None))[0]
)
df_songs['label_mbid'] = df_songs['musicbrainz_recording_mbid'].map(
    lambda x: recording_to_labels.get(x, (None, None))[1]
)

matched_labels = df_songs['label'].notna().sum()
print(f"Labels matched: {matched_labels:,}/{len(df_songs):,} ({matched_labels/len(df_songs)*100:.1f}%)")

df_songs.head()


Unique recording MBIDs to look up: 27,267
Labels matched: 24,660/38,383 (64.2%)


,title,name,performer_pre_normalized,original_performer_name,original_performer_name_normalized,peak_pos,wks_on_chart,first_charting_year,musicbrainz_artist_id,musicbrainz_recording_mbid,label,label_mbid
0,It's All In The Game,tommy edwards,Tommy Edwards,NaN,NaN,1,22,1958,0b1a0ca5-52cb-4d1d-8344-746f905ae115,7cf06b8f-a898-4f2c-a589-a98af8af6001,[Not Now Music],[b454b444-eb75-46ed-b400-19e85d8e1c64]
1,It's Only Make Believe,conway twitty,Conway Twitty,NaN,NaN,1,21,1958,a3c60d26-90d6-4788-ba9b-a89693fc396d,4123e40f-d667-4634-919d-6e1d77a19934,[Remasters Workshop],[e55f5360-a4d0-43a1-b84d-f69b8510f3c6]
2,Little Star,the elegants,The Elegants,NaN,NaN,1,17,1958,91919ad2-80cb-4077-bbb3-2803fa129fab,e15d4722-47a7-4874-99f6-285529503569,[Club Records],[9019dc41-f533-4c25-81b1-35fc98531903]
3,Nel Blu Dipinto Di Blu (Volaré),domenico modugno,Domenico Modugno,NaN,NaN,1,16,1958,b8e60837-e646-4f18-8f92-27d1173511b0,858ea2b4-7cc8-4a13-aa93-a3a5a73e71fd,[Warner Music Italy],[30d5acae-ae46-4689-9a91-99293a55e95e]
4,Poor Little Fool,ricky nelson,Ricky Nelson,NaN,NaN,1,11,1958,28d0c272-4d51-4c24-b31f-e20aac2ba7de,100b7f30-1519-427a-adc2-2b0d6a737db1,[Chartbuster Karaoke],[4c44731f-f69b-4bac-b396-064980e894eb]


In [24]:
# This cell defines the major label sets for Warner Music Group, Sony Music
# Entertainment, and Universal Music Group, including all known subsidiaries
# and imprints, plus commonly occurring MusicBrainz label names that were
# missing from the original sets.

# Warner Music Group and subsidiaries
warner_labels = {
    'Warner', 'Atlantic', 'Atlantic Records', 'Elektra Records', 'Parlophone Records',
    'Warner Records', 'Atlantic Music Group', '1st & 15th Entertainment', 'All Money In',
    'Artist Partner Group', 'Asylum Records', 'Atco Records', 'Avang Music',
    'Big Beat Records', 'Big Tree Records', 'Boulevard Boyz', "Bread Winners' Association",
    'Canvasback Music', 'Cat Records', 'Guwop Enterprises', '808 Mafia Records', 'CBE',
    'Chop Shop Records', 'Taylor Gang Entertainment', 'Tha Mvmnt', 'Sniper Gang Records',
    'Chopper City Records', 'Cotillion Records', 'CTE World', 'Custard Records',
    'Eardrum Records', 'Esperanza Records', 'First Priority Music', 'Fort Knocks Entertainment',
    'Full Surface Records', 'F-Stop Music', 'FNC Entertainment', 'Grind Or Die Label',
    'Human Re Sources', 'Heavy Muscle, LLC', 'LaSalle Records', 'Little David Records',
    'Luke Records', 'Nice Life Recording Company', 'Owsla', 'Nest', 'Photo Finish Records',
    'Poe Boy Music Group', 'Rebel Rock Entertainment', 'Sandlot Records', "Spinnin' Records",
    'Congo Records', 'CONTROVERSIA', 'Dharma Worldwide', 'DOORN Records', 'Heartfeldt Records',
    'Hysteria', 'Kryteria Records', 'Maxximize Records', 'Mentalo Music',
    'Musical Freedom Records / AFTR:HRS', 'Night Service Only', 'OZ Records', 'SOURCE',
    'SINPHONY', "Spinnin' Deep", "Spinnin' NEXT", "Spinnin' Premium", "Spinnin' Records Asia",
    "Spinnin' Talent Pool", 'SPRS', 'Tonco Tone', 'Wall Recordings', 'Stone Flower Records',
    'TAG Recordings', 'Top Stop Music', 'UpFront Records', 'Vortex Records', 'X5 Music Group',
    'Cake Jamz Records', '300 Entertainment', 'EastWest Records America', 'Black Cement',
    'DCD2 Records', 'DTA Records', 'Fueled by Ramen', 'Low Country Sound',
    'Roadrunner Records', 'Public Consumption Recording Co.', '12Tone Music Group',
    '143 Records', 'A&E Records', 'Action Theory Records', 'Artery Recordings',
    'Beluga Heights', 'BME Recordings', 'Defiant Records', 'Doghouse Records',
    'Facultad de Némea', 'Festival Mushroom Records', 'Globetown Music', 'Helium 3',
    'Ice Age Entertainment', 'Jet Life Recordings', 'Loveway Records', 'One Town Media',
    'Machine Shop Recordings', 'Malpaso Records', 'Maverick Records', 'Nonesuch Records',
    'Slash Records', 'Parlophone', 'Perezcious Music', 'Perfecto Records', 'Playmaker Music',
    'Public Broadcasting Service', 'Reprise Records', 'RuffNation Records', 'Sire Records',
    'SM Entertainment', 'Red Weather Records', 'Stateside Records', 'Top Rank Records',
    'Parlophone Records Ltd.', 'Chrysalis Records', 'EMI Columbia Records', 'EMI Records',
    'Harvest Records', 'Regal Recordings', 'Regal Zonophone', 'Virgin Records',
    'FFRR Records', 'Kling Klang Produkt', 'Marv Music', 'Rhino Entertainment',
    'Bearsville Records', 'Del-Fi Records', 'Eleven: A Music Company', 'Jubilee Records',
    'Rhino Records', 'Roulette Records', 'TK Records', '10K Projects', '3CG Records',
    '5 Minute Walk', '679 Recordings', 'Acony Records', 'Adrenaline Music', 'Alligator Records',
    'Alma Records', 'Alta Note Records', 'Anzic Records', 'Artist First', 'Arhoolie Records',
    'Avitone Records', 'Bar/None Records', 'Barsuk Records', 'Beggars Group',
    'Bieler Bros. Records', 'Blind Pig Records', 'Bloodshot Records', 'Blix Street Records',
    'Blisslife Records', 'Blue Corn Music', 'Blue Horizon', 'Bolero Records',
    'Born & Bred Records', 'Brash Records', 'Breakbeat Science Records', 'Bright Antenna',
    'Canyon Records', 'Carpark Records', 'Cavity Search Records', 'CDBaby', 'Chesky Records',
    'Chime Entertainment', 'Chiyun Records', 'Comedy Central Records', 'Compass Records',
    'Cordless Recordings', 'Courgette Records', 'Crunchy Frog Records', 'Curb Records',
    'David Lynch Music Company', 'Dcide Records', 'Decaydance Records', 'Dentro Recordings',
    'DimeRock Records', 'Divendy Records', 'Domino Recording Company', 'Downtown Records',
    'Dualtone Records', 'East West Records', 'Eco Records', 'el Music Group',
    'Everfine Records', 'Eyeball Records', 'Ferret Music', 'Kontor Records', 'Funzalo Records',
    'Guelmi Music', 'Hellcat Records', 'I Scream Records', 'Igualdade Vision Records',
    'Internet Money', 'InVogue Records', 'Jemp Records', 'Lightyear Entertainment',
    'Mascom Records', 'Masquerade Recordings', 'Macklemore Inc.', 'Matador Records',
    'Mayhem Records', 'Manifesto Records', 'Merge Records', 'Metade Note Records',
    'Milan Records', 'Misfits Records', 'Metropolis Records', 'Music Branding', 'Mute Records',
    'Napalm Records', 'Nervous Records', 'Nettwerk', 'Nexchapter', 'No Filter Records',
    'Nuclear Blast', 'Nudgie Records', 'Om Records', 'Phony Records', 'Polyvinyl Record Company',
    'Projekt Records', 'PRT Records', 'Pye Records', 'Punahele Records', 'Relapse Records',
    'Restless Records', 'Rhymesayers Entertainment', 'Rhythmbank Entertainment', 'Rise Records',
    'Rock Ridge Music', 'Rykodisc Records', 'Saddle Creek Records', 'Sanctuary Records',
    'Bronze Records', 'Castle Records', 'CMC International', 'GWR Records',
    'Sideonedummy Records', 'Skeleton Crew', 'Slugfest Records', 'Stony Plain Records',
    'Strictly Rhythm Records', 'Sub Pop Records', 'Surfdog Records', 'Tee Pee Records',
    'Teleprompt Records', 'The Control Group', 'Thirsty Ear Recordings', 'Thrill Jockey Records',
    'TML Entertainment', 'Tommy Boy Entertainment', 'Touch & Go Records', 'Troubleman Unlimited',
    'Twenty Two Recordings', 'Ubiquity Records', 'Upstream Records', 'Vice Records',
    'VMG Recordings', 'Voltra Music Group', 'Warcon Enterprises', 'WaterTower Music',
    'Wichita Records', 'Word Entertainment', 'DaySpring Records', 'Myrrh Records',
    'Word Records', 'You Entertainment', 'Arts Music', 'Sesame Street Records',
    'First Night Records', 'Mattel Music', 'Sh-K-Boom Records', 'Warner Classics',
    'Warner Classics Records', 'Erato Records', 'Teldec Records', 'Elektra Nonesuch Records',
    'Finlandia Records', 'Lontano Records', 'Warner Apex', 'Warner Elatus',
    'Warner Fonit Records', 'NVC Arts Records', 'Warner Music Nashville',
    'Atlantic Records Nashville', 'Warner Records Nashville', 'Elektra Records Nashville',
    'Marcus Music', 'Balkan Electro', 'Warner Music HK', 'Mariann Grammofon AB',
    'Warner Music UK', 'Warner Records UK', 'Atlantic Records UK', 'Asylum Records UK',
    'Rhino UK', 'London Records', '14th Floor Records', 'Warner Music Sweden',
    'Warner Music Spain', 'Warner Music Portugal', 'Parlophone Portugal', 'Warner Music Poland',
    'EMI Music Poland', 'Polskie Nagrania', 'Warner Music Norway', 'K. Dahl Eftf.',
    'Warner Music Benelux', 'Atlantic Records Benelux', 'Warner Music Belgium',
    'Warner Music Netherlands', 'Warner Music Central Europe', 'Atlantic Records Germany',
    'Warner Music Austria', 'Warner Music Germany', 'Warner Music Hungary',
    'Warner Music Switzerland', 'UDR Records', 'Warner Music Italy', 'ADA Italy',
    'Warner Records Italy', 'Atlantic Records Italy', 'Warner Music Ireland',
    'Warner Music Greece', 'Warner Music France', 'ADA France', 'Parlophone France',
    'Delabel', 'Décibels Productions', 'Elektra France', 'Atlantic Records France',
    'Nous Productions', 'Warner Music Finland', 'Finnlevy', 'Fazer Records', 'Evidence',
    'Helsinki Music Company', 'Warner Music Baltics', 'Warner Music Denmark', 'Medley',
    'Warner Music Czech Republic', '1967 Ltd', 'Must Destroy Records', 'The Beats',
    'Warner Strategic Marketing', 'Warner Music Romania', 'Roton', 'Global Records',
    'Warner Music Africa', 'Warner Music Australia', 'Warner Music New Zealand',
    'Illegal Musik', 'Gold Typhoon', 'Warner Music China', 'Warner Music Hong Kong',
    'F Records', 'Whet Drops', 'Warner Music India', 'Maati', 'Sky Digital',
    'Always Music Global', 'Indie Music Label', 'Desh Music', 'Divo', '91 NORTH RECORDS',
    'Warner Music Indonesia', 'Hemagita Records', 'Virgo Ramayana', 'Alfa Records',
    'Indo Semar Sakti', 'Warner Music Japan', 'A.K.A. Records', "A'zip Music",
    'Atlantic Records Japan', 'Cube Loves Music!', 'Defstar Records', 'Dream Machine',
    'East West Records Japan', 'Entrance', 'Fueled By Mentaiko', 'Futurista', 'Garland',
    'Jupiter', 'Minami', 'Moon', 'N.A.T.', 'No Label Music', 'Organon', 'Pasion Record!',
    'Photon', 'Planets', 'Real Note', 'River Way', 'Reprise Records Japan',
    'Rhino Records Japan', 'Roadrunner Records Japan', 'Samurai Rock', 'Sprouse', 'Taco Records',
    'Trinitas', 'Unborde', 'Vybe', 'Warner Sunset', 'WEA Japan', 'World Art',
    'Warner Music Korea', 'Brand New Music', 'Keystone Entertainment', 'Vitamin Entertainment',
    'WS Entertainment', 'MPLIFY Records', 'Warner Music Malaysia', 'Ramada Records',
    'Black Hat Cat Records', 'Zamrud', 'Roslan Aziz Productions', 'Warner Music Middle East',
    'Warner Music Philippines', 'Music Colony Records', 'PressPlay.ph', 'Sora Music Group',
    'Warner Music Pakistan', 'Giraffe Music', 'Warner Music Singapore',
    'Music Street', 'Play Music', 'Warner Music Taiwan', 'UFO Records', 'Warner Music Thailand',
    'D-Day', 'Muser', 'Wayfer Record', 'Onpa Music', 'Warner Music Turkey',
    'Warner Music Vietnam', 'SpaceSpeakers', 'Warner Music Argentina', 'Warner Music Brasil',
    'Banguela Records', 'Caravela Records', 'EH Brasil', 'Furacão 2000', 'Selo Chantecler',
    'Selo Continental', 'WEA Discos', 'RW Produtora', 'Warner Strategic Marketing Brazil',
    'Warner Music Chile', 'Warner Music Colombia', 'Warner Music Mexico', 'Warner Music Perú',
    'Warner Music Canada', 'Warner Music Group',
    # Commonly occurring MusicBrainz labels added
    'Warner Bros. Records', 'Rhino', 'Elektra', 'Warner Bros.',
}

# Sony Music Entertainment and subsidiaries
sony_labels = {
    'Sony', 'Sony Music', 'Columbia Records', 'RCA Records', 'Epic Records', 'Arista Records',
    'Legacy Recordings', 'Sony Music Latin', 'Sony Masterworks', 'Provident Label Group',
    'Sony Classical', '30th Century Records', 'Alamo Records', 'Hypnotize Minds', 'i am OTHER',
    'Reach Records', 'Disruptor Records', 'Erskine Records', 'Cactus Jack Records',
    'Dreamcatcher Company', 'Battery Records', 'Vested In Culture', 'Freebandz', 'MJJ Music',
    'will.i.am Music Group', 'Jive Records', 'RCA Inspiration', 'Chris Brown Entertainment',
    'GospoCentric Records', 'Orange Records', 'Keep Cool Records', 'KQ Entertainment',
    'Verity Records', 'ByStorm Entertainment', 'Nappy Boy Entertainment', 'Francis Records',
    'Polo Grounds Music', 'ASAP Mob', 'Kemosabe Records', 'Roswell Records', 'House of Iona',
    'Sony Music Nashville', 'Columbia Nashville', 'RCA Records Nashville', 'Brentwood Records',
    'Benson Records', 'Essential Records', 'Flicker Records', 'Beach Street Records',
    'Reunion Records', 'Provident Special Markets', 'Milan Records', 'Playbill Records',
    'Bluebird Records', 'Masterworks Broadway', 'Okeh Records', 'Portrait Records',
    'RCA Red Seal Records', 'Sony Classical Records', 'Odyssey Records', 'J Records',
    'Windham Hill Records', 'LaFace Records', 'Buddah Records', 'Kama Sutra Records',
    'Philadelphia International Records', 'Philles Records', 'The Orchard', 'RED Music',
    'Black River Entertainment', 'Blind Pig Records', 'Big Future Records',
    'Century Media Records', 'Another Century Records', 'Inside Out Music',
    'Cinematic Music Group', 'Frenchkiss Records', 'JDub Records', 'Louder Than Life',
    'Madison Gate Records', 'Odd Future Records', 'Shrapnel Records', 'SIEGUS',
    'Soundx3 Records', 'TVT Records', 'Valley Entertainment', 'Xanadu Records',
    'Sony Music Publishing', 'Associated Production Music', 'Sonoton', 'Bruton Music',
    'Cezame Music', 'Hard and Kosinus', 'KPM Music', 'Extreme Music',
    'Remote Control Productions', 'Bleeding Fingers Music', 'Hickory Records', 'Dial Records',
    '4 Star Records', 'Challenge Records', 'Sony Music UK', 'Columbia Records UK',
    'RCA Label Group', 'Epic Records UK', 'Relentless Records', 'Ministry of Sound',
    'Music For Nations', '5K Records', 'Black Butter Records', 'Dream Life Records',
    'Insanity Records', 'Magic Star', 'Sony Music Masterworks', 'Masterworks',
    'Deutsche Harmonia Mundi', 'Robots + Humans', "Since '93", 'Sony Commercial Group',
    'Sony Music Nashville UK', 'WEAREBLK', 'AWAL', 'Sony Music Entertainment Japan',
    'Aniplex', 'Ariola Japan', 'Defstar Records', 'Echoes', 'Epic Records Japan',
    'Gr8! Records', 'Ki/oon Music', 'Mastersix Foundation', 'Peanuts Worldwide', 'Red Cafe',
    'Sacra Music', 'SME Records', 'Sony Music Distribution', 'Sony Music Records',
    'Studioseven Recordings', 'Time Records', 'Santa Anna Label Group', 'OVO Sound',
    'Sony Music Africa', 'Sony Music Argentina', 'Sony Music Australia', 'Ariola Records',
    'Sony Music Brasil', 'Amigo Records', 'Austro Music', 'Blast Stage Records', 'Inbraza',
    'LG7', 'Phonomotor Records', 'Som Livre', 'SLAP', 'The Orchard Brasil', 'Sony Music Canada',
    'Ratas Music Group', 'GUN Records', 'X-Cell Records', '604 Records',
    'Sony Music Chile & Bolivia', 'Sony Music China', 'Sony Music Colombia',
    'Sony Music Czech Republic', 'Supraphon', 'Sony Music Denmark', 'Mermaid Records',
    'Sony Music France', 'Arista France', 'RCA Records France', 'Sony Music Germany',
    'Ariola Records GmbH', 'Sony Music Greece', 'Sony Music Entertainment Hong Kong',
    'O.U.R. Works', 'Green Music', 'Sony Music India', 'Sony Music South',
    'Sony Music Indonesia', 'Sony Music Ireland', 'Sony Music Italy', 'Sony Music Japan',
    'Sony Music Korea', 'GLG', 'Pop Music', 'Sony Music Malaysia Sdn. Bhd.',
    '410 Sony Music Records', 'Sony Music Mexico', 'Sony Music Middle East',
    'Sony Music Pakistan', 'Sony Music Norway', 'Sony Music New Zealand',
    'Sony Music Philippines', 'Sony Music Peru', 'Offmute', 'Waterwalk Records',
    'Sony Music Entertainment Poland', 'Sony Music Portugal', 'Azteca Music',
    'Sony Music Romania', 'Sony Music Russia', 'Sony Music Singapore', 'Sony Music Spain',
    'Sony Music Sweden', 'Razzia Records', 'Sony Music Taiwan', 'Sony Music Thailand',
    'Love Is', 'Bakery Music', 'Sony Music Vietnam', 'Sony Music Entertainment Africa/South Africa',
    'South African Recordings', 'Select Records', 'Albert Productions', 'Aware Records',
    'Buppu Records', 'Chrome Entertainment', 'Constellation Records', 'Cooking Vinyl',
    'Danger Crue Records', 'Fair Trade Services', 'Hostess Entertainment', 'In the Name Of',
    'Ivory Music and Video', 'J Storm', 'Kobalt Music Group', 'Konnect Entertainment',
    'Megaforce Records', 'MCI', 'Midas Music', 'Mirror Society', 'MODHAUS', 'Mr. 305 Inc.',
    'Napalm Records', 'Nick Records', 'CBS Records', 'Palm Tree Records', 'Shout! Factory',
    'Starship Entertainment', 'Stmpd Rcrds', 'SPV GmbH', 'Robbins Entertainment',
    'Round Hill Music', 'Black Hill Records', 'Dolores Recordings', 'Woah Dad!',
    'Drakkar Entertainment', 'Tratore', 'TOP Media', 'Top Stop Music', 'Ultra Records',
    'EEM Records', 'WM Entertainment', 'WWE Music Group',
    'American Recordings', 'Daft Life', 'Def Jam Recordings', 'Jam Master Jay Records',
    'Fever Records', 'C2 Records', 'LBW Entertainment', 'Ruffhouse Records',
    'Ovum Recordings', 'Loud Records', 'DIW Records', 'Vinyl Solution',
    'So So Def Recordings', 'Skint Records', 'Skyblaze Recordings', 'Startime International',
    'Burgundy Records', 'Chaos Recordings', 'Oriole Records', 'Private-I Records',
    'Bobcat Records', 'Razor Sharp Records', 'Bad Boy Records', 'Epic Soundtrax',
    "Cold Chillin' Records", 'Immortal Records', '550 Music', 'WTG Records', 'Work Group',
    'Ode Records', 'Ruthless Records', 'Hidden Beach Recordings', 'Caribou Records',
    'Tabu Records', 'Jet Records', 'Tuff City Records', 'Nemperor Records', 'Sony Wonder',
    'Zomba Group of Companies', 'Internal Affairs', 'EBUL', 'Jive Electro',
    'Volcano Entertainment', 'Zoo Entertainment', 'Scotti Brothers Records',
    'Capricorn Records', 'RED Distribution', 'Living Music', 'Private Music', 'Novus Records',
    'RCA Camden', 'RCA Gold Seal', 'RCA Victrola', 'Vik Records', 'Rowdy Records',
    'Profile Records', 'Kinetic Records', 'Logic Records', 'BMG Kidz', 'Jim Henson Records',
    'Bell Records', 'Go-Feet Records', 'BMG Entertainment', 'ECM Records', 'Imago Records',
    'V2 Records', 'Gee Street Records', "Junior Boy's Own", 'Beyond Music', 'BMG Heritage',
    'Sanctuary Records', 'Sanctuary Urban', 'BMG Funhouse', 'Amiga',
    'BMG Records Pilipinas/Musiko Records', 'Arte Nova Classics', 'Conifer Records',
    'Deutsche Schallplatten Berlin (DSB)', 'Hansa Records', 'Arista Nashville',
    'Epic Records Nashville', 'Sony S2', 'Silvertone Records', 'Phonogenic Records',
    # Commonly occurring MusicBrainz labels added
    'Epic', 'Sony Music', 'OVO Sound',
}

# Universal Music Group and subsidiaries
universal_labels = {
    'Universal', 'Virgin', 'Abbey Road Studios', 'Capitol Music Group', 'Decca Records',
    'Def Jam Recordings', 'Deutsche Grammophon', 'EMI', 'Interscope-Geffen-A&M',
    'Island Records', 'Mercury Records', 'Motown', 'Polydor Records', 'Republic Records',
    'Universal Records', 'Verve Label Group', 'Virgin Records', 'Interscope Records',
    'Geffen Records', 'A&M Records', 'Aftermath Entertainment', 'AWGE',
    'Billion Dollar Baby Entertainment', 'Blackground Records', 'Cloud 9', 'Tunechi',
    'Lost Highway Records', 'Shady Records', 'PGLang', 'F2 Records', 'Maloof Music',
    'Mosley Music Group', 'Top Dawg Entertainment', 'Dvpper Music', 'Darkroom Records',
    'Downtown Records', 'Dreamville Records', 'Konvict Kulture', 'Streamline Records',
    'Opium', 'Suretone Records', 'Weapons of Mass Entertainment', 'Zone 4',
    'Tropical Records', 'KIDinaKORNER', 'HYBE', 'Big Hit Music', 'Source Music',
    'Belift Lab', 'Pledis Entertainment', 'KOZ Entertainment', 'ADOR',
    'Cinematic Music Group', '222 Records', 'EarsDrummers Entertainment', 'N.E.E.T. Recordings',
    "Hits Since '87", 'Division1', 'Creative Arts Recordings', 'Play the Beat Entertainment',
    'YG Entertainment', 'SMG Music', 'Astralwerks', 'Blue Note Records', 'Block Entertainment',
    'Capitol Records', 'Capitol Christian Music Group', 'Capitol Christian Music Group Canada',
    'Deep Well Records', 'Deep Well Music Publishing', 'Get Money Gang Entertainment',
    'G-Unit Records', 'G-Note Records', 'G-Unit Film & Television', 'Harvest Records',
    'Hi or Hey Records', 'Manhattan Records', 'Matriarch Records', 'Metamorphosis Music',
    'Priority Records', 'Quality Control Music', 'RedOne Records', 'Siren Records',
    '10 Records', 'Mi5 Recordings', 'Originals, Inc.', 'RMG Music Group',
    'Str8 Wired Entertainment', 'Pinegrove Records', 'The Trak Kartel Records', 'Tracklist',
    'Sonorous Entertainment', 'Fader Label', 'Republic Corps', 'Republic Records',
    'American Recordings', 'Aware Records', 'BME Recordings', 'Brushfire Records',
    'Casablanca Records', 'Cash Money Content', 'Cash Money Records', 'Cranchynine Records',
    'Galactic Records', 'Hybe Corporation', 'Indie Pop Music', 'John Varvatos Records',
    'JYP Entertainment', 'Lava Records', 'Motown Records', 'Motown Gospel',
    'Next Plateau Entertainment', 'Photo Finish Records', 'Rich Gang', 'Rowdy Records',
    'RiteorWrongKVH Entertainment', 'Schoolboy Records', 'Serjical Strike Records',
    'So What the Fuss Records', 'SRC Records', 'Trukfit', 'Universal Arabic Music',
    'Uptown Records', 'XO Records', 'YMCMB Apparel', 'Young Money Entertainment',
    "4th & B'way Records", 'Safehouse Records', 'Super Records', 'Teen Island',
    'Rave Music Entertainment', 'Amusement Records', '410 Records', 'ARTium Recordings',
    'Desert Storm Records', 'Radio Killa Records', 'Black Star Records', 'Circa 13 Music',
    'Darksyde Productions Inc Records', 'Music Corporation of America', 'MCA Nashville',
    'Mercury Nashville', 'Capitol Records Nashville', 'EMI Records Nashville',
    'Decca Nashville', 'Universal Music Latin Entertainment', 'Universal Music Latino',
    'Fonovisa Records', 'Interscope Miami', 'Disa Records', 'Capitol Latin', 'Machete Music',
    'Ivy Queen Musa Sound Corporation', 'All Star Records', 'Flow Music',
    'Illegal Life Records', 'Mas Flow Inc.', 'Sangre Nueva Music', 'VI Music',
    'Verve Label Group', 'Decca Music Group', 'Verve Music Group', 'Fontana Records',
    'Brunswick Records', 'Decca Classics', 'Decca Vision', 'Philips Records', 'Verve Records',
    'GRP Records', 'Impulse! Records', 'Verve Forecast Records', 'Virgin Music Group',
    'Caroline Records', 'Fiction Records', 'mtheory Artist Partnerships', 'Integral',
    'A24 Music', 'Virgin Music Africa', 'Virgin Music Argentina', 'Virgin Music Australia',
    'Virgin Music Benelux', 'Virgin Music Brasil', 'Virgin Music Chile', 'Virgin Music France',
    'Virgin Music Germany', 'Virgin Music Italy', 'Virgin Music Japan', 'Virgin Music Latin US',
    'Virgin Music Mexico', 'Virgin Music Nigeria', 'Virgin Music Nordics', 'Virgin Music Norway',
    'Virgin Music Spain', 'Virgin Music Sweden', 'Virgin Music UK', 'Virgin Music US',
    'SONO Music Group', 'Icon Entertainment', 'Lorimar Records',
    'Universal Music Publishing Group', 'Criterion Music Corporation',
    'Universal Production Music', 'Associated Production Music', 'Sonoton', 'Bruton Music',
    'Cezame Music', 'Hard and Kosinus', 'Universal Music Enterprises', 'Universal Chronicles',
    'UM3', 'T-Boy Records', 'Urban Legends', 'Thump Records', 'Mercury Studios',
    'Eagle Rock Entertainment', 'Eagle Records', 'Armoury Records', 'Eagle Rock Productions',
    'Eagle-i Music', 'Universal Music UK', 'Universal Music TV',
    'The Universal Music Record Label', 'Argo Records', 'EmArcy Records', 'Xploded Music',
    'Clubland TV', 'Now 70s', 'Now 80s', 'Now 90s', 'Globe: Soundtrack and Score',
    'Universal Music Recordings', 'Calderstone Productions Limited', 'Charisma Records',
    'Purple Records', 'ZTT Records', 'Island EMI Label Group', 'Island Records UK',
    'EMI Records', 'EMI North', 'Vertigo Records', 'Blackened Recordings', 'Ensign Records',
    'Motown UK', 'PMR Records', 'Positiva Records', 'Polydor Label Group',
    'Fascination Records', 'A&M Records UK', 'Rolling Stones Records', '0207 Def Jam',
    'Capitol Records UK', 'Universal Music France', 'AZ Records', 'Barclay Records',
    'ECM New Series', 'MCA Records France', 'Casablanca Records France',
    'PM:AM Recordings France', 'Island Def Jam', 'Virgin Records France',
    'Capitol Label Services', 'Universal Licensing Music', 'Polydor Records France', 'Mosaert',
    'Universal Music Jazz France', 'Universal Music Japan', 'Universal J', 'Perfume Records',
    'Asse!! Records', 'RX-Records', 'UJ', 'Project K&P', 'Over the Top Records',
    'Project TJ', 'Universal Sigma', 'Double Joy International', 'DCT Records',
    'Augusta Records', 'U-Cube', 'Yoshimoto Universal Tunes', 'Virgin Music', 'Kinashi Records',
    'Eastworld', 'Go Good Records', 'Mercury Records Tokyo',
    'Holo-n', 'E-Sum Records', 'Tunes Tracks', 'Universal Classics and Jazz', 'Universal D',
    'Universal International', 'Thunderball 667', 'Pachinko Records',
    'Universal Strategic Marketing Japan', 'Zero-A', 'Universal Music Sweden',
    'Capitol Music Group Sweden', 'Capitol Records Sweden', 'Kavalkad', 'Lionheart Music',
    'SoFo Records', 'Virgin Records Sweden', 'Polar Music', 'Pope Records', 'Sonet Records',
    'Stockholm Records', 'Minos EMI', 'Universal Music Africa', 'Universal Music Cameroon',
    "Universal Music Côte d'Ivoire", 'Motown Gospel Africa', 'Universal Music Nigeria',
    'Universal Music Senegal', 'Universal Music South Africa', 'Def Jam Africa',
    'Universal Music Andina', 'Universal Music Argentina', 'Universal Music Austria',
    'Universal Music Azerbaijan', 'Universal Music Australia', 'Casablanca Records Australia',
    'Dew Process', 'EMI Recorded Music Australia', 'Golden Era Records',
    'Island Records Australia', 'Lost Highway Records Australia', 'Neon Records', 'Of Leisure',
    'Universal Music Belgium', 'PIAS Recordings', 'Different Recordings', 'Le Label', 'Urban',
    'Harmonia Mundi', 'Inertia', 'Jazz Village', 'World Village', 'V2 Records',
    'ARS Entertainment', 'Universal Music Brazil', 'Arsenal Music', 'Phonomotor Records',
    'Universal Music Bulgaria', 'Universal Music Canada', 'Maison Barclay Canada',
    'Universal Music Chile', 'Universal Music Greater China', 'Capitol Records China',
    'PolyGram China', 'Republic Records China', 'EMI China', 'Universal Music China',
    'Universal Music Hong Kong', 'PolyGram Records', 'Cinepoly Records',
    'Go East Entertainment', "What's Music", 'EMI Music Hong Kong', 'Brave Music',
    'Universal Music Taiwan', 'EMI Taiwan', 'Universal Music Colombia',
    'Universal Music Czech Republic', 'Universal Music Denmark', 'MBO Group',
    'Copenhagen Records', 'Universal Music Estonia', 'Universal Music Finland',
    'Spinefarm Records', 'Johanna Kustannus', 'Inka Entertainment', 'Island Records Finland',
    'Universal Music Germany', 'Universal Music Domestic Division', 'Urban Records',
    'Universal Music International Division', 'Universal Music Classics and Jazz',
    'Koch Universal Music', 'Universal Music Strategic Marketing',
    'Universal Music Family Entertainment', 'Polydor/Island', 'Vertigo/Capitol', 'Electrola',
    'Odeon Records', 'Universal Music Georgia', 'Universal Music Hungary',
    'Universal Music India', 'Def Jam India', 'VYRL Originals', 'VYRL South', 'VYRL Punjabi',
    'VYRL Haryanvi', 'VYRL Bhojpuri', 'Indiea Records', 'Oriental Star Agencies',
    'Desi Melodies', 'Virgin Records India', 'Mad For Mussic', 'Universal Music Indonesia',
    'BLⱯCKBOARD', 'Solid Records', 'GP Records', 'Massive Music Entertainment',
    'Wonderland Records', 'Dominion Records', 'RFAS Music', 'Juni Records',
    'Def Jam Indonesia', 'PreachJa Records', 'Universal Music Ireland', 'Universal Music Italy',
    'Island Records Italy', 'Capitol Records Italy', 'Virgin Records Italy',
    'Universal Music Korea', 'Universal Music Kazakhstan', 'Universal Music Kyrgyzstan',
    'Universal Music Latvia', 'Universal Music Lithuania', 'Universal Music Malaysia',
    'Def Jam Malaysia', 'EMI Malaysia', 'Rumpun Records', 'XO House', 'Universal Music MENA',
    'Universal Music Mexico', 'EMI Mexico', 'Universal Music Morocco',
    'Universal Music Netherlands', 'Universal Music New Zealand', 'Universal Music Norway',
    'Jazzland Recordings', 'UMG Philippines', 'Island Records Philippines',
    'Def Jam Philippines', 'Republic Records Philippines', 'EMI Records Philippines',
    'Universal Music Polska', 'Magic Records', 'Universal Music Portugal',
    'Universal Music Peru', 'Universal Music Romania', 'MediaPro Music',
    'Def Jam Recordings Romania', 'Universal Music Serbia', 'Universal Music Singapore',
    'Def Jam Asia', 'Astralwerks Asia', 'Universal Music Spain', 'Vale Music',
    'Universal Music Switzerland', 'Universal Music Tajikistan', 'Universal Music Thailand',
    'Def Jam Recordings Thailand', 'Universal Music Turkey', 'Universal Music Turkmenistan',
    'Universal Music Ukraine', 'Universal Music Vietnam', 'Deutsche Grammophon Production Music',
    'Flexitracks', 'KTV', 'Nuvotone', 'Sonic Beat Records', 'Volta Music', 'EMI Arabia',
    'Soutelphan', 'Alam El Phan', 'Relax-in International', 'Farasan', 'Rotana Records',
    'EMI Music Argentina', 'EMI Music Brazil', 'Discos Copacabana', 'EMI-Jangada',
    'EMI Music Chile', 'EMIDISC', 'EMI Europe Generic', 'EMI Gold', 'EMI Music Ireland',
    'EMI Music Hungary', 'EMI Music Finland', 'EMI Music Germany', 'Awake Sounds',
    'Capitol Music Japan', 'Eastworld', 'Express', 'Foozay Music', 'i-Dance',
    'Reservotion Records', 'SakuraStar Records', 'SoundTown', 'Suite Supuesto!',
    'TM Factory', 'Unlimited Records', 'EMI Music Pakistan', 'EMI Music South Africa',
    'GramCo', 'Q-Productions', 'Reliquias', 'Bravado', 'PolyGram Entertainment', '10:22PM',
    'Victor Victor Worldwide', 'Field Trip Recordings', 'ABKCO Records',
    'Cameo-Parkway Records', 'Back Lot Music', 'Because Music', 'Ed Banger Records',
    'Phantasy', 'London Recordings', 'Factory Records', 'BMG Rights Management',
    'Big Machine Label Group', 'Big Machine Records', 'Valory Music',
    'Nashville Harbor Records & Entertainment', 'Nash Icon Music', 'Disney Music Group',
    'Walt Disney Records', 'Hollywood Records', 'Hollywood BASIC', 'Fox Music',
    'Rachanok Entertainment', 'DMG Nashville', 'Yuehua Entertainment',
    'Redcherry Entertainment', 'Walt Disney South', 'Concord', 'Craft Recordings',
    'Wind-up Records', 'Fania Records', 'El Cartel Records', 'Razor & Tie',
    'Varèse Sarabande', 'Varèse Vintage', 'DIVA Records', 'Earth Hertz Records',
    'EDGEOUT Records', 'Famous Records', 'Hater Gang Records', 'MDM Recordings',
    'Ministry of Sound Australia', 'Polyversal', 'Primary Wave', 'Primary Wave Records',
    'Sun Records', 'Roc Nation', 'StarRoc', 'Takeover Roc Nation', 'Round Hill Music',
    'Warrior Records', 'Paramount', 'Comedy Central Records', 'VP Records', 'Actually Music',
    'Dreamus', 'Mirage Records', 'Sena', 'Helicon Records', 'Prime Music',
    'Ukrainian Records', 'Tuff Gong', 'YG Plus', "Musica Studio's", 'Off the Record',
    'Pen Island Records', 'RS Music', 'JVR Music', '20th Century Fox Records',
    '89 Arrogance Recordings', 'ABC Records', 'Aladdin Records', 'Almo Sounds',
    'A&M Octone Records', 'Angel Records', 'Argo Records', 'Atlanta Artists', 'Axis Records',
    'Back Beat Records', 'Backstreet Records', 'Biv 10 Records', 'Blue Thumb Records',
    'BTM Music Group Inc.', 'Cadet Records', 'Chess Records', 'Cherrytree Records',
    'Chrysalis Records', 'Coral Records', 'Cube Entertainment', 'Cypress Records',
    'De-Lite Records', 'Decca Broadway', 'DGC Records', 'Diamond Star Entertainment',
    'Dolton Records', 'Dot Records', 'DreamWorks Records', 'Duke Records', 'Dunhill Records',
    'E.G. Records', 'EMI America Records', 'EMI Records India', 'EMI USA', 'Enigma Records',
    'Fantasy Records', 'Freedom', 'Friends Keep Secrets', 'Gasoline Alley Records',
    'Hut Records', 'Impact Records', 'Imperial Records', 'Infinity Records', 'Ingrooves',
    'Intercord Tonträger', 'I.R.S. Records', 'The Island Def Jam Music Group', 'Kapp Records',
    'Laurie Records', 'Liberty Records', 'Loud Records', 'Mad Love Records', 'MCA Records',
    'Mediarts Records', 'MGM Records', 'Minit Records', 'Nocturne Records', 'Octone Records',
    'Odeon Records', 'Outpost Recordings', 'Pacific Jazz Records', 'Paramount Records',
    'Parrot Records', 'Peacock Records', 'Perspective Records', 'Pina Records',
    'Polydor India', 'Polygram India / Music india', 'Radioactive Records',
    'Regal Zonophone Records', 'Revue Records', 'RMM Records & Video', 'Roc-A-Fella Records',
    'SBK Records', 'Shelter Records', 'Smash Records', 'Spinnup', 'Star Trak Entertainment',
    'Silas Records', 'Tennman Records', 'Uni Records', 'United Artists Records',
    'Universal Music Israel', 'Universal Music Russia', 'Universal Republic Records',
    'Universal Motown Republic Group', 'Universal Motown Records', 'UZI Suicide',
    'Vendetta Records', 'Virgin EMI Records', 'Virgin Music', 'Vocalion Records',
    'Wing Records', 'WY Records',
    # Commonly occurring MusicBrainz labels added
    'Interscope Records', 'Cash Money Records', 'Alamo Records', 'Young Money',
    'Geffen Records', 'Universal Music', 'Universal Music TV', 'Aftermath Entertainment',
    'Capitol Records Nashville', 'Walt Disney Records', 'Quality Control Music',
    'Shady Records', 'Universal Music Group', 'MGM Records', 'Hollywood Records',
    'Decca Records', 'Universal Republic Records', 'Generation Now', 'Boominati Worldwide',
    'Hip‐O Records', 'Maybach Music Group', 'Grade A Productions', 'Freebandz Entertainment',
}

# Master list of all major label terms
all_major_labels = warner_labels | sony_labels | universal_labels

print(f"Warner Music Group labels: {len(warner_labels)}")
print(f"Sony Music labels: {len(sony_labels)}")
print(f"Universal Music Group labels: {len(universal_labels)}")
print(f"Total unique major label terms: {len(all_major_labels)}")


Warner Music Group labels: 468
Sony Music labels: 339
Universal Music Group labels: 668
Total unique major label terms: 1446


In [25]:
# This code classifies each song's label data into major label categories using
# exact set membership plus a small set of safe manual keywords for each major.
# The auto-generated keyword expansion was dropped due to false positives.
# Adds 7 new columns after 'label':
#   - major_label: bool, whether any label is a known major label
#   - major_label_names: list of matched major label names
#   - other_label: bool, whether any label is NOT a major label
#   - other_label_names: list of non-major label names
#   - warner: bool, Warner Music Group affiliation
#   - universal: bool, Universal Music Group affiliation
#   - sony: bool, Sony Music Entertainment affiliation

import pandas as pd

# Drop existing classification columns if present from a previous run
cols_to_drop = ['major_label', 'major_label_names', 'other_label', 'other_label_names',
                'warner', 'universal', 'sony']
df_songs = df_songs.drop(columns=[c for c in cols_to_drop if c in df_songs.columns])

def contains_keyword(label, keywords):
    label_lower = label.lower()
    return any(keyword in label_lower for keyword in keywords)

def add_label_classification_columns(df, label_sets):
    df = df.copy()

    major_label_bool, major_label_names_list = [], []
    other_label_bool, other_label_names_list = [], []
    warner_bool, universal_bool, sony_bool = [], [], []

    for labels in df['label']:
        if not isinstance(labels, list):
            labels = []

        major_labels = [l for l in labels if l in label_sets['all_major']]
        other_labels = [l for l in labels if l not in label_sets['all_major']]

        is_warner = any(
            l in label_sets['warner'] or contains_keyword(l, ['warner', 'atlantic'])
            for l in labels
        )
        is_universal = any(
            l in label_sets['universal'] or
            contains_keyword(l, ['universal', 'interscope', 'capitol', 'virgin'])
            for l in labels
        )
        is_sony = any(
            l in label_sets['sony'] or
            contains_keyword(l, ['sony', 'columbia', 'rca', 'epic', 'arista'])
            for l in labels
        )

        major_label_bool.append(len(major_labels) > 0)
        major_label_names_list.append(major_labels if major_labels else None)
        other_label_bool.append(len(other_labels) > 0)
        other_label_names_list.append(other_labels if other_labels else None)
        warner_bool.append(is_warner)
        universal_bool.append(is_universal)
        sony_bool.append(is_sony)

    label_col_idx = df.columns.get_loc('label')

    new_columns = pd.DataFrame({
        'major_label':       major_label_bool,
        'major_label_names': major_label_names_list,
        'other_label':       other_label_bool,
        'other_label_names': other_label_names_list,
        'warner':            warner_bool,
        'universal':         universal_bool,
        'sony':              sony_bool,
    }, index=df.index)

    df = pd.concat([
        df.iloc[:, :label_col_idx + 1],
        new_columns,
        df.iloc[:, label_col_idx + 1:]
    ], axis=1)

    return df

label_sets = {
    'warner':    warner_labels,
    'sony':      sony_labels,
    'universal': universal_labels,
    'all_major': all_major_labels
}

df_songs = add_label_classification_columns(df_songs, label_sets)

print(f"df_songs shape: {df_songs.shape}")
print(f"\nMajor label:  {df_songs['major_label'].sum():,} ({df_songs['major_label'].mean()*100:.1f}%)")
print(f"Warner:       {df_songs['warner'].sum():,} ({df_songs['warner'].mean()*100:.1f}%)")
print(f"Universal:    {df_songs['universal'].sum():,} ({df_songs['universal'].mean()*100:.1f}%)")
print(f"Sony:         {df_songs['sony'].sum():,} ({df_songs['sony'].mean()*100:.1f}%)")
print(f"Other label:  {df_songs['other_label'].sum():,} ({df_songs['other_label'].mean()*100:.1f}%)")

df_songs.head()


df_songs shape: (38383, 19)

Major label:  10,425 (27.2%)
Warner:       2,944 (7.7%)
Universal:    6,556 (17.1%)
Sony:         4,714 (12.3%)
Other label:  16,300 (42.5%)


,title,name,performer_pre_normalized,original_performer_name,original_performer_name_normalized,peak_pos,wks_on_chart,first_charting_year,musicbrainz_artist_id,musicbrainz_recording_mbid,label,major_label,major_label_names,other_label,other_label_names,warner,universal,sony,label_mbid
0,It's All In The Game,tommy edwards,Tommy Edwards,NaN,NaN,1,22,1958,0b1a0ca5-52cb-4d1d-8344-746f905ae115,7cf06b8f-a898-4f2c-a589-a98af8af6001,[Not Now Music],False,None,True,[Not Now Music],False,False,False,[b454b444-eb75-46ed-b400-19e85d8e1c64]
1,It's Only Make Believe,conway twitty,Conway Twitty,NaN,NaN,1,21,1958,a3c60d26-90d6-4788-ba9b-a89693fc396d,4123e40f-d667-4634-919d-6e1d77a19934,[Remasters Workshop],False,None,True,[Remasters Workshop],False,False,False,[e55f5360-a4d0-43a1-b84d-f69b8510f3c6]
2,Little Star,the elegants,The Elegants,NaN,NaN,1,17,1958,91919ad2-80cb-4077-bbb3-2803fa129fab,e15d4722-47a7-4874-99f6-285529503569,[Club Records],False,None,True,[Club Records],False,False,False,[9019dc41-f533-4c25-81b1-35fc98531903]
3,Nel Blu Dipinto Di Blu (Volaré),domenico modugno,Domenico Modugno,NaN,NaN,1,16,1958,b8e60837-e646-4f18-8f92-27d1173511b0,858ea2b4-7cc8-4a13-aa93-a3a5a73e71fd,[Warner Music Italy],True,[Warner Music Italy],False,None,True,False,False,[30d5acae-ae46-4689-9a91-99293a55e95e]
4,Poor Little Fool,ricky nelson,Ricky Nelson,NaN,NaN,1,11,1958,28d0c272-4d51-4c24-b31f-e20aac2ba7de,100b7f30-1519-427a-adc2-2b0d6a737db1,[Chartbuster Karaoke],False,None,True,[Chartbuster Karaoke],False,False,False,[4c44731f-f69b-4bac-b396-064980e894eb]


In [26]:
# Save df_songs to CSV as checkpoint
# Saves current state of df_songs including all columns added so far:
# title, performer, musicbrainz_artist_id, musicbrainz_recording_mbid,
# label, label_mbid, major_label, major_label_group, and all Billboard metadata

df_songs.to_csv(
    '/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_songs.csv',
    index=False
)

print(f"Saved df_songs: {len(df_songs):,} rows × {len(df_songs.columns)} columns")
print(f"\nColumns: {list(df_songs.columns)}")


Saved df_songs: 38,383 rows × 19 columns

Columns: ['title', 'name', 'performer_pre_normalized', 'original_performer_name', 'original_performer_name_normalized', 'peak_pos', 'wks_on_chart', 'first_charting_year', 'musicbrainz_artist_id', 'musicbrainz_recording_mbid', 'label', 'major_label', 'major_label_names', 'other_label', 'other_label_names', 'warner', 'universal', 'sony', 'label_mbid']


In [ ]:
# Add genre_tags and tags columns from MusicBrainz for each recording in df_songs
# Single query using LEFT JOIN on genre table and array_agg FILTER to split results:
#   - genre_tags: official MusicBrainz genres only (tag name appears in genre table)
#   - tags: all other positively-voted tags (aliases, hybrid labels, format, instrumentation, etc.)
# Both lists are ordered by vote count descending within each recording

import psycopg2
import psycopg2.extras
import threading

DB_PARAMS = {
    'dbname': 'musicbrainz_db',
    'user': 'musicbrainz',
    'password': 'musicbrainz',
    'host': 'localhost',
    'port': 5432
}

matched_mbids = (
    df_songs['musicbrainz_recording_mbid']
    .dropna()
    .unique()
    .tolist()
)
print(f"Recording MBIDs to look up tags for: {len(matched_mbids):,}")

genre_results = {}
tag_results = {}
error_holder = [None]
done_event = threading.Event()

def run_query():
    try:
        conn = psycopg2.connect(**DB_PARAMS)
        conn.autocommit = True
        cur = conn.cursor()

        cur.execute("SET jit = off")
        cur.execute("SET statement_timeout = '300s'")

        cur.execute("""
            CREATE TEMP TABLE tag_input_mbids (recording_mbid uuid)
            ON COMMIT PRESERVE ROWS
        """)

        psycopg2.extras.execute_values(
            cur,
            "INSERT INTO tag_input_mbids VALUES %s",
            [(mbid,) for mbid in matched_mbids],
            page_size=1000
        )

        cur.execute("CREATE INDEX ON tag_input_mbids (recording_mbid)")
        cur.execute("ANALYZE tag_input_mbids")

        cur.execute("""
            SELECT
                im.recording_mbid::text,
                array_agg(t.name ORDER BY rt.count DESC)
                    FILTER (WHERE g.name IS NOT NULL) AS genre_tags,
                array_agg(t.name ORDER BY rt.count DESC)
                    FILTER (WHERE g.name IS NULL)     AS tags
            FROM tag_input_mbids im
            JOIN recording r      ON r.gid       = im.recording_mbid
            JOIN recording_tag rt ON rt.recording = r.id
            JOIN tag t            ON t.id         = rt.tag
            LEFT JOIN genre g     ON g.name       = t.name
            WHERE rt.count > 0
            GROUP BY im.recording_mbid
        """)

        for mbid, genres, tags in cur.fetchall():
            if genres:
                genre_results[mbid] = genres
            if tags:
                tag_results[mbid] = tags

        cur.close()
        conn.close()

    except Exception as e:
        error_holder[0] = e
    finally:
        done_event.set()

t = threading.Thread(target=run_query, daemon=True)
t.start()
done_event.wait()

if error_holder[0]:
    raise error_holder[0]

print(f"Genre tags found for: {len(genre_results):,} / {len(matched_mbids):,} recordings "
      f"({len(genre_results)/len(matched_mbids)*100:.1f}%)")
print(f"Other tags found for: {len(tag_results):,} / {len(matched_mbids):,} recordings "
      f"({len(tag_results)/len(matched_mbids)*100:.1f}%)")

df_songs['genre_tags'] = df_songs['musicbrainz_recording_mbid'].map(genre_results)
df_songs['tags'] = df_songs['musicbrainz_recording_mbid'].map(tag_results)

genre_tagged = df_songs['genre_tags'].notna().sum()
tag_tagged   = df_songs['tags'].notna().sum()
print(f"\nRows with genre_tags: {genre_tagged:,} / {len(df_songs):,} ({genre_tagged/len(df_songs)*100:.1f}%)")
print(f"Rows with tags:       {tag_tagged:,} / {len(df_songs):,} ({tag_tagged/len(df_songs)*100:.1f}%)")


Recording MBIDs to look up tags for: 27,267
Genre tags found for: 5,841 / 27,267 recordings (21.4%)
Other tags found for: 3,538 / 27,267 recordings (13.0%)

Rows with genre_tags: 6,310 / 38,383 (16.4%)
Rows with tags:       3,856 / 38,383 (10.0%)

Sample:


KeyError: "['performer'] not in index"

In [33]:
# Rename tag columns to clarify they are recording-level tags from MusicBrainz
df_songs = df_songs.rename(columns={
    'genre_tags': 'song_genre_tags_musicbrainz',
    'tags':       'song_tags_musicbrainz'
})
print(df_songs.columns.tolist())


['title', 'name', 'performer_pre_normalized', 'original_performer_name', 'original_performer_name_normalized', 'peak_pos', 'wks_on_chart', 'first_charting_year', 'musicbrainz_artist_id', 'musicbrainz_recording_mbid', 'label', 'major_label', 'major_label_names', 'other_label', 'other_label_names', 'warner', 'universal', 'sony', 'label_mbid', 'song_genre_tags_musicbrainz', 'song_tags_musicbrainz']


In [34]:
# Add album_genre_tags_musicbrainz column to df_songs
# Queries official MusicBrainz genre tags applied at the release group (album) level
# Chain: recording → track → medium → release → release_group → release_group_tag → tag → genre
# A recording may appear on multiple release groups; tags from all are merged and deduplicated
# Adds column: album_genre_tags_musicbrainz (list of genre strings, or NaN if none found)

import psycopg2
import psycopg2.extras
import threading

DB_PARAMS = {
    'dbname': 'musicbrainz_db',
    'user': 'musicbrainz',
    'password': 'musicbrainz',
    'host': 'localhost',
    'port': 5432
}

matched_mbids = (
    df_songs['musicbrainz_recording_mbid']
    .dropna()
    .unique()
    .tolist()
)
print(f"Recording MBIDs to look up album genre tags for: {len(matched_mbids):,}")

results = {}
error_holder = [None]
done_event = threading.Event()

def run_query():
    try:
        conn = psycopg2.connect(**DB_PARAMS)
        conn.autocommit = True
        cur = conn.cursor()

        cur.execute("SET jit = off")
        cur.execute("SET statement_timeout = '300s'")

        cur.execute("""
            CREATE TEMP TABLE album_tag_input_mbids (recording_mbid uuid)
            ON COMMIT PRESERVE ROWS
        """)

        psycopg2.extras.execute_values(
            cur,
            "INSERT INTO album_tag_input_mbids VALUES %s",
            [(mbid,) for mbid in matched_mbids],
            page_size=1000
        )

        cur.execute("CREATE INDEX ON album_tag_input_mbids (recording_mbid)")
        cur.execute("ANALYZE album_tag_input_mbids")

        cur.execute("""
            SELECT
                im.recording_mbid::text,
                array_agg(DISTINCT t.name)
                    FILTER (WHERE g.name IS NOT NULL) AS album_genre_tags
            FROM album_tag_input_mbids im
            JOIN recording r          ON r.gid            = im.recording_mbid
            JOIN track tr             ON tr.recording      = r.id
            JOIN medium m             ON m.id              = tr.medium
            JOIN release rel          ON rel.id            = m.release
            JOIN release_group rg     ON rg.id             = rel.release_group
            JOIN release_group_tag rgt ON rgt.release_group = rg.id
            JOIN tag t                ON t.id              = rgt.tag
            LEFT JOIN genre g         ON g.name            = t.name
            WHERE rgt.count > 0
            GROUP BY im.recording_mbid
        """)

        for mbid, tags in cur.fetchall():
            if tags:
                results[mbid] = tags

        cur.close()
        conn.close()

    except Exception as e:
        error_holder[0] = e
    finally:
        done_event.set()

t = threading.Thread(target=run_query, daemon=True)
t.start()
done_event.wait()

if error_holder[0]:
    raise error_holder[0]

print(f"Album genre tags found for: {len(results):,} / {len(matched_mbids):,} recordings "
      f"({len(results)/len(matched_mbids)*100:.1f}%)")

df_songs['album_genre_tags_musicbrainz'] = df_songs['musicbrainz_recording_mbid'].map(results)

tagged = df_songs['album_genre_tags_musicbrainz'].notna().sum()
print(f"Rows with album_genre_tags_musicbrainz: {tagged:,} / {len(df_songs):,} ({tagged/len(df_songs)*100:.1f}%)")

print(f"\nSample:")
sample = df_songs[df_songs['album_genre_tags_musicbrainz'].notna()][
    ['title', 'original_performer_name', 'song_genre_tags_musicbrainz', 'album_genre_tags_musicbrainz']
].sample(10)
print(sample.to_string())


Recording MBIDs to look up album genre tags for: 27,267
Album genre tags found for: 17,512 / 27,267 recordings (64.2%)
Rows with album_genre_tags_musicbrainz: 18,819 / 38,383 (49.0%)

Sample:
                           title       original_performer_name        song_genre_tags_musicbrainz                                                                                                                                                                                                                                                                                              album_genre_tags_musicbrainz
5088        Comin' On Too Strong                           NaN                                NaN                                                                                                                                                                                                                                                                                                           

In [35]:
# Apply major genre categorization to df_songs using MusicBrainz genre tags
# Creates three columns:
#   - song_major_genre_category_musicbrainz:      categories from song-level genre tags
#   - album_major_genre_category_musicbrainz:     categories from album-level genre tags
#   - aggregate_major_genre_category_musicbrainz: union of both (deduplicated, sorted)
# Input columns contain Python lists (not strings), so the function handles lists directly

import pandas as pd

# Complete mapping dictionary
musicbrainz_to_major_genre = {

    # ROCK
    'rock': 'Rock',
    'rock and roll': 'Rock',
    'classic rock': 'Rock',
    'hard rock': 'Rock',
    'soft rock': 'Rock',
    'arena rock': 'Rock',
    'heartland rock': 'Rock',
    'mainstream rock': 'Rock',
    'southern rock': 'Rock',
    'roots rock': 'Rock',
    'swamp rock': 'Rock',
    'pub rock': 'Rock',
    'garage rock': 'Rock',
    'garage rock revival': 'Rock',
    'garage psych': 'Rock',
    'psychedelic rock': 'Rock',
    'psychedelic': 'Rock',
    'acid rock': 'Rock',
    'raga rock': 'Rock',
    'heavy psych': 'Rock',
    'progressive rock': 'Rock',
    'progressive': 'Rock',
    'prog rock': 'Rock',
    'symphonic rock': 'Rock',
    'symphonic prog': 'Rock',
    'crossover prog': 'Rock',
    'krautrock': 'Rock',
    'space rock': 'Rock',
    'glam rock': 'Rock',
    'glam': 'Rock',
    'blues rock': 'Rock',
    'boogie rock': 'Rock',
    'surf rock': 'Rock',
    'surf': 'Rock',
    'instrumental rock': 'Rock',
    'post-rock': 'Rock',
    'noise rock': 'Rock',
    'alternative rock': 'Rock',
    'indie rock': 'Rock',
    'grunge': 'Rock',
    'post-grunge': 'Rock',
    'britpop': 'Rock',
    'post-britpop': 'Rock',
    'shoegaze': 'Rock',
    'dream pop': 'Rock',
    'jangle pop': 'Rock',
    'paisley underground': 'Rock',
    'neo-psychedelia': 'Rock',
    'lo-fi': 'Rock',
    'slacker rock': 'Rock',
    'geek rock': 'Rock',
    'merseybeat': 'Rock',
    'beat music': 'Rock',
    'freakbeat': 'Rock',
    'mod': 'Rock',
    'nederbeat': 'Rock',
    'nederpop': 'Rock',
    'frat rock': 'Rock',
    'yacht rock': 'Rock',
    'aor': 'Rock',
    'industrial rock': 'Rock',
    'electronic rock': 'Rock',
    'dance-rock': 'Rock',
    'folk rock': 'Rock',
    'celtic rock': 'Rock',
    'medieval rock': 'Rock',
    'country rock': 'Rock',
    'reggae rock': 'Rock',
    'tropical rock': 'Rock',
    'stoner rock': 'Rock',
    'desert rock': 'Rock',
    'occult rock': 'Rock',
    'rockabilly': 'Rock',
    'psychobilly': 'Rock',
    'acoustic rock': 'Rock',
    'piano rock': 'Rock',
    'vocal surf': 'Rock',
    'jam band': 'Rock',
    'steampunk': 'Rock',
    'rock opera': 'Rock',
    'comedy rock': 'Rock',
    'j-rock': 'Rock',
    'visual kei': 'Rock',
    'gothic': 'Rock',
    'gothic rock': 'Rock',
    'funk rock': 'Rock',
    'rap rock': 'Rock',
    'slowcore': 'Rock',
    'canterbury scene': 'Rock',
    'neue deutsche welle': 'Rock',

    # ELECTRONIC/DANCE
    'electronic': 'Electronic/Dance',
    'electronica': 'Electronic/Dance',
    'dance': 'Electronic/Dance',
    'club': 'Electronic/Dance',
    'edm': 'Electronic/Dance',
    'rave': 'Electronic/Dance',
    'house': 'Electronic/Dance',
    'deep house': 'Electronic/Dance',
    'tech house': 'Electronic/Dance',
    'progressive house': 'Electronic/Dance',
    'electro house': 'Electronic/Dance',
    'acid house': 'Electronic/Dance',
    'afro house': 'Electronic/Dance',
    'funky house': 'Electronic/Dance',
    'vocal house': 'Electronic/Dance',
    'french house': 'Electronic/Dance',
    'italo house': 'Electronic/Dance',
    'dutch house': 'Electronic/Dance',
    'euro house': 'Electronic/Dance',
    'tropical house': 'Electronic/Dance',
    'big room house': 'Electronic/Dance',
    'festival progressive house': 'Electronic/Dance',
    'ambient house': 'Electronic/Dance',
    'techno': 'Electronic/Dance',
    'ambient techno': 'Electronic/Dance',
    'trance': 'Electronic/Dance',
    'progressive trance': 'Electronic/Dance',
    'vocal trance': 'Electronic/Dance',
    'hard trance': 'Electronic/Dance',
    'big room trance': 'Electronic/Dance',
    'euro-trance': 'Electronic/Dance',
    'goa trance': 'Electronic/Dance',
    'psytrance': 'Electronic/Dance',
    'psybient': 'Electronic/Dance',
    'dubstep': 'Electronic/Dance',
    'brostep': 'Electronic/Dance',
    'future garage': 'Electronic/Dance',
    'uk garage': 'Electronic/Dance',
    'uk funky': 'Electronic/Dance',
    'drum and bass': 'Electronic/Dance',
    'jungle': 'Electronic/Dance',
    'liquid funk': 'Electronic/Dance',
    'neurofunk': 'Electronic/Dance',
    'drill and bass': 'Electronic/Dance',
    'jungle terror': 'Electronic/Dance',
    'breakbeat': 'Electronic/Dance',
    'breakbeat hardcore': 'Electronic/Dance',
    'breakcore': 'Electronic/Dance',
    'big beat': 'Electronic/Dance',
    'happy hardcore': 'Electronic/Dance',
    'ambient': 'Electronic/Dance',
    'dark ambient': 'Electronic/Dance',
    'drone': 'Electronic/Dance',
    'downtempo': 'Electronic/Dance',
    'trip hop': 'Electronic/Dance',
    'chillout': 'Electronic/Dance',
    'chillwave': 'Electronic/Dance',
    'idm': 'Electronic/Dance',
    'glitch': 'Electronic/Dance',
    'glitch hop': 'Electronic/Dance',
    'wonky': 'Electronic/Dance',
    'electro': 'Electronic/Dance',
    'disco': 'Electronic/Dance',
    'nu disco': 'Electronic/Dance',
    'italo disco': 'Electronic/Dance',
    'euro-disco': 'Electronic/Dance',
    'electro-disco': 'Electronic/Dance',
    'hi-nrg': 'Electronic/Dance',
    'freestyle': 'Electronic/Dance',
    'synthwave': 'Electronic/Dance',
    'vaporwave': 'Electronic/Dance',
    'future bass': 'Electronic/Dance',
    'bass music': 'Electronic/Dance',
    'trap edm': 'Electronic/Dance',
    'moombahton': 'Electronic/Dance',
    'jersey club': 'Electronic/Dance',
    'ghettotech': 'Electronic/Dance',
    'eurodance': 'Electronic/Dance',
    'eurobeat': 'Electronic/Dance',
    'italo dance': 'Electronic/Dance',
    'bubblegum dance': 'Electronic/Dance',
    'new beat': 'Electronic/Dance',
    'electroclash': 'Electronic/Dance',
    'new rave': 'Electronic/Dance',
    'electro swing': 'Electronic/Dance',
    'complextro': 'Electronic/Dance',
    'future rave': 'Electronic/Dance',
    'industrial': 'Electronic/Dance',
    'ebm': 'Electronic/Dance',
    'martial industrial': 'Electronic/Dance',
    'dark wave': 'Electronic/Dance',
    'witch house': 'Electronic/Dance',
    'futurepop': 'Electronic/Dance',
    'berlin school': 'Electronic/Dance',
    'minimal synth': 'Electronic/Dance',
    'progressive electronic': 'Electronic/Dance',
    'bitpop': 'Electronic/Dance',
    'ethereal wave': 'Electronic/Dance',
    'hip house': 'Electronic/Dance',
    'alternative dance': 'Electronic/Dance',
    'folktronica': 'Electronic/Dance',
    'funktronica': 'Electronic/Dance',
    'electropunk': 'Electronic/Dance',
    'indietronica': 'Electronic/Dance',
    'chiptune': 'Electronic/Dance',
    'italo-disco': 'Electronic/Dance',
    'breaks': 'Electronic/Dance',
    'hardstyle': 'Electronic/Dance',
    'dungeon synth': 'Electronic/Dance',
    'lolicore': 'Electronic/Dance',
    'leftfield': 'Electronic/Dance',

    # POP
    'pop': 'Pop',
    'pop rock': 'Pop',
    'pop soul': 'Pop',
    'dance-pop': 'Pop',
    'electropop': 'Pop',
    'teen pop': 'Pop',
    'bubblegum pop': 'Pop',
    'power pop': 'Pop',
    'baroque pop': 'Pop',
    'art pop': 'Pop',
    'chamber pop': 'Pop',
    'indie pop': 'Pop',
    'noise pop': 'Pop',
    'sunshine pop': 'Pop',
    'sophisti-pop': 'Pop',
    'brill building': 'Pop',
    'space age pop': 'Pop',
    'europop': 'Pop',
    'j-pop': 'Pop',
    'k-pop': 'Pop',
    'q-pop': 'Pop',
    'alternative pop': 'Pop',
    'ambient pop': 'Pop',
    'psychedelic pop': 'Pop',
    'progressive pop': 'Pop',
    'avant-garde pop': 'Pop',
    'traditional pop': 'Pop',
    'schlager': 'Pop',
    'yé-yé': 'Pop',
    'pop yeh-yeh': 'Pop',
    'shibuya-kei': 'Pop',
    'ballad': 'Pop',
    'synth-pop': 'Pop',
    'new romantic': 'Pop',
    'new wave': 'Pop',
    'hyperpop': 'Pop',
    'glitch pop': 'Pop',
    'hypnagogic pop': 'Pop',
    'bedroom pop': 'Pop',
    'pop punk': 'Punk/Hardcore',
    'pop rap': 'Hip Hop/Rap',
    'emo pop': 'Punk/Hardcore',
    'twee pop': 'Pop',

    # HIP HOP/RAP
    'hip hop': 'Hip Hop/Rap',
    'rap': 'Hip Hop/Rap',
    'east coast hip hop': 'Hip Hop/Rap',
    'west coast hip hop': 'Hip Hop/Rap',
    'southern hip hop': 'Hip Hop/Rap',
    'dirty south': 'Hip Hop/Rap',
    'gangsta rap': 'Hip Hop/Rap',
    'g-funk': 'Hip Hop/Rap',
    'hardcore hip hop': 'Hip Hop/Rap',
    'conscious hip hop': 'Hip Hop/Rap',
    'political hip hop': 'Hip Hop/Rap',
    'alternative hip hop': 'Hip Hop/Rap',
    'experimental hip hop': 'Hip Hop/Rap',
    'abstract hip hop': 'Hip Hop/Rap',
    'instrumental hip hop': 'Hip Hop/Rap',
    'drumless hip hop': 'Hip Hop/Rap',
    'boom bap': 'Hip Hop/Rap',
    'jazz rap': 'Hip Hop/Rap',
    'trap': 'Hip Hop/Rap',
    'drill': 'Hip Hop/Rap',
    'chicago drill': 'Hip Hop/Rap',
    'jersey drill': 'Hip Hop/Rap',
    'detroit trap': 'Hip Hop/Rap',
    'grime': 'Hip Hop/Rap',
    'crunk': 'Hip Hop/Rap',
    'crunkcore': 'Hip Hop/Rap',
    'snap': 'Hip Hop/Rap',
    'hyphy': 'Hip Hop/Rap',
    'cloud rap': 'Hip Hop/Rap',
    'emo rap': 'Hip Hop/Rap',
    'plugg': 'Hip Hop/Rap',
    'rage': 'Hip Hop/Rap',
    'miami bass': 'Hip Hop/Rap',
    'memphis rap': 'Hip Hop/Rap',
    'chopped and screwed': 'Hip Hop/Rap',
    'horrorcore': 'Hip Hop/Rap',
    'nerdcore': 'Hip Hop/Rap',
    'underground hip hop': 'Hip Hop/Rap',
    'industrial hip hop': 'Hip Hop/Rap',
    'turntablism': 'Hip Hop/Rap',
    'lo-fi hip hop': 'Hip Hop/Rap',
    'ratchet music': 'Hip Hop/Rap',
    'digicore': 'Hip Hop/Rap',
    'country rap': 'Hip Hop/Rap',
    'christian hip hop': 'Hip Hop/Rap',

    # R&B/SOUL/FUNK
    'r&b': 'R&B/Soul/Funk',
    'rhythm & blues': 'R&B/Soul/Funk',
    'contemporary r&b': 'R&B/Soul/Funk',
    'alternative r&b': 'R&B/Soul/Funk',
    'neo soul': 'R&B/Soul/Funk',
    'quiet storm': 'R&B/Soul/Funk',
    'soul': 'R&B/Soul/Funk',
    'southern soul': 'R&B/Soul/Funk',
    'northern soul': 'R&B/Soul/Funk',
    'deep soul': 'R&B/Soul/Funk',
    'psychedelic soul': 'R&B/Soul/Funk',
    'progressive soul': 'R&B/Soul/Funk',
    'blue-eyed soul': 'R&B/Soul/Funk',
    'smooth soul': 'R&B/Soul/Funk',
    'chicago soul': 'R&B/Soul/Funk',
    'philly soul': 'R&B/Soul/Funk',
    'motown': 'R&B/Soul/Funk',
    'doo-wop': 'R&B/Soul/Funk',
    'funk': 'R&B/Soul/Funk',
    'p-funk': 'R&B/Soul/Funk',
    'synth funk': 'R&B/Soul/Funk',
    'boogie': 'R&B/Soul/Funk',
    'go-go': 'R&B/Soul/Funk',
    'new orleans r&b': 'R&B/Soul/Funk',
    'minneapolis sound': 'R&B/Soul/Funk',
    'electro-funk': 'R&B/Soul/Funk',
    'hip hop soul': 'R&B/Soul/Funk',
    'new jack swing': 'R&B/Soul/Funk',
    'trap soul': 'R&B/Soul/Funk',

    # COUNTRY/AMERICANA
    'country': 'Country/Americana',
    'classic country': 'Country/Americana',
    'traditional country': 'Country/Americana',
    'contemporary country': 'Country/Americana',
    'country pop': 'Country/Americana',
    'country soul': 'Country/Americana',
    'outlaw country': 'Country/Americana',
    'alternative country': 'Country/Americana',
    'progressive country': 'Country/Americana',
    'neo-traditional country': 'Country/Americana',
    'bakersfield sound': 'Country/Americana',
    'nashville sound': 'Country/Americana',
    'honky tonk': 'Country/Americana',
    'western swing': 'Country/Americana',
    'bluegrass': 'Country/Americana',
    'progressive bluegrass': 'Country/Americana',
    'bro-country': 'Country/Americana',
    'red dirt': 'Country/Americana',
    'urban cowboy': 'Country/Americana',
    'country yodeling': 'Country/Americana',
    'yodeling': 'Country/Americana',
    'americana': 'Country/Americana',
    'cajun': 'Country/Americana',
    'zydeco': 'Country/Americana',
    'swamp pop': 'Country/Americana',
    'country blues': 'Blues',
    'country folk': 'Folk',
    'country gospel': 'Gospel/Christian/Religious',
    'texas country': 'Country/Americana',
    'old-time': 'Country/Americana',
    'western': 'Country/Americana',

    # FOLK
    'folk': 'Folk',
    'contemporary folk': 'Folk',
    'folk pop': 'Pop',
    'progressive folk': 'Folk',
    'psychedelic folk': 'Folk',
    'chamber folk': 'Folk',
    'avant-folk': 'Folk',
    'anti-folk': 'Folk',
    'indie folk': 'Folk',
    'neofolk': 'Folk',
    'neo-acoustic': 'Folk',
    'stomp and holler': 'Folk',
    'singer-songwriter': 'Folk',
    'ambient americana': 'Folk',
    'british folk rock': 'Rock',
    'folk punk': 'Punk/Hardcore',
    'folk metal': 'Metal',
    'irish folk': 'Folk',
    'celtic new age': 'Easy Listening/Vocal',
    'freak folk': 'Folk',
    'contra': 'Folk',
    'alternative folk': 'Folk',

    # JAZZ
    'jazz': 'Jazz',
    'bebop': 'Jazz',
    'hard bop': 'Jazz',
    'post-bop': 'Jazz',
    'cool jazz': 'Jazz',
    'free jazz': 'Jazz',
    'avant-garde jazz': 'Jazz',
    'jazz fusion': 'Jazz',
    'jazz rock': 'Jazz',
    'jazz pop': 'Jazz',
    'contemporary jazz': 'Jazz',
    'smooth jazz': 'Jazz',
    'swing': 'Jazz',
    'big band': 'Jazz',
    'dixieland': 'Jazz',
    'acid jazz': 'Jazz',
    'spiritual jazz': 'Jazz',
    'vocal jazz': 'Jazz',
    'digital fusion': 'Jazz',
    'jazz-funk': 'Jazz',
    'soul jazz': 'Jazz',
    'close harmony': 'Jazz',
    'modal jazz': 'Jazz',
    'nu jazz': 'Jazz',
    'free improvisation': 'Jazz',

    # BLUES
    'blues': 'Blues',
    'electric blues': 'Blues',
    'chicago blues': 'Blues',
    'delta blues': 'Blues',
    'acoustic blues': 'Blues',
    'texas blues': 'Blues',
    'electric texas blues': 'Blues',
    'hill country blues': 'Blues',
    'classic blues': 'Blues',
    'piano blues': 'Blues',
    'british blues': 'Blues',
    'british rhythm & blues': 'Blues',
    'soul blues': 'Blues',
    'boogie-woogie': 'Blues',

    # METAL
    'metal': 'Metal',
    'heavy metal': 'Metal',
    'thrash metal': 'Metal',
    'death metal': 'Metal',
    'black metal': 'Metal',
    'power metal': 'Metal',
    'speed metal': 'Metal',
    'progressive metal': 'Metal',
    'folk metal': 'Metal',
    'gothic metal': 'Metal',
    'symphonic metal': 'Metal',
    'glam metal': 'Metal',
    'nu metal': 'Metal',
    'metalcore': 'Metal',
    'deathcore': 'Metal',
    'grindcore': 'Metal',
    'goregrind': 'Metal',
    'melodic death metal': 'Metal',
    'melodic black metal': 'Metal',
    'atmospheric black metal': 'Metal',
    'depressive black metal': 'Metal',
    'blackgaze': 'Metal',
    'symphonic black metal': 'Metal',
    'brutal death metal': 'Metal',
    'technical death metal': 'Metal',
    'death-doom metal': 'Metal',
    'funeral doom metal': 'Metal',
    'traditional doom metal': 'Metal',
    'doom metal': 'Metal',
    'sludge metal': 'Metal',
    'stoner metal': 'Metal',
    'groove metal': 'Metal',
    'industrial metal': 'Metal',
    'neoclassical metal': 'Metal',
    'nwobhm': 'Metal',
    'mathcore': 'Metal',
    'djent': 'Metal',
    'kawaii metal': 'Metal',
    'alternative metal': 'Metal',
    'crossover thrash': 'Metal',
    'noisecore': 'Metal',
    'post-metal': 'Metal',
    'funk metal': 'Metal',
    'rap metal': 'Metal',
    'rapcore': 'Metal',
    'trap metal': 'Metal',
    'mashcore': 'Metal',
    'melodic metalcore': 'Metal',
    'avant-garde metal': 'Metal',
    'christian metal': 'Metal',
    'progressive metalcore': 'Metal',
    'deathgrind': 'Metal',
    'electro-industrial': 'Metal',
    'electronicore': 'Metal',
    'southern metal': 'Metal',
    'us power metal': 'Metal',

    # PUNK/HARDCORE
    'punk': 'Punk/Hardcore',
    'punk rock': 'Punk/Hardcore',
    'hardcore punk': 'Punk/Hardcore',
    'post-hardcore': 'Punk/Hardcore',
    'anarcho-punk': 'Punk/Hardcore',
    'd-beat': 'Punk/Hardcore',
    'skate punk': 'Punk/Hardcore',
    'ska punk': 'Punk/Hardcore',
    'horror punk': 'Punk/Hardcore',
    'glam punk': 'Punk/Hardcore',
    'art punk': 'Punk/Hardcore',
    'proto-punk': 'Punk/Hardcore',
    'dance-punk': 'Punk/Hardcore',
    'emo': 'Punk/Hardcore',
    'emocore': 'Punk/Hardcore',
    'screamo': 'Punk/Hardcore',
    'emoviolence': 'Punk/Hardcore',
    'grebo': 'Punk/Hardcore',
    'post-punk': 'Punk/Hardcore',
    'post-punk revival': 'Punk/Hardcore',
    'no wave': 'Punk/Hardcore',
    'midwest emo': 'Punk/Hardcore',
    'punk blues': 'Punk/Hardcore',
    'melodic hardcore': 'Punk/Hardcore',
    'garage punk': 'Punk/Hardcore',
    'cowpunk': 'Punk/Hardcore',
    'easycore': 'Punk/Hardcore',
    'crust punk': 'Punk/Hardcore',
    'celtic punk': 'Punk/Hardcore',

    # LATIN
    'latin': 'Latin',
    'latin ballad': 'Latin',
    'latin pop': 'Latin',
    'latin rock': 'Latin',
    'salsa': 'Latin',
    'bachata': 'Latin',
    'mambo': 'Latin',
    'son cubano': 'Latin',
    'tango': 'Latin',
    'mariachi': 'Latin',
    'norteño': 'Latin',
    'corrido tumbado': 'Latin',
    'regional mexicano': 'Latin',
    'reggaeton': 'Latin',
    'electro latino': 'Latin',
    'canción melódica': 'Latin',
    'tex-mex': 'Latin',
    'trap latino': 'Latin',
    'bossa nova': 'Latin',
    'choro': 'Latin',
    'mpb': 'Latin',
    'latin jazz': 'Jazz',
    'banda sinaloense': 'Latin',
    'ranchera': 'Latin',
    'bolero': 'Latin',
    'merengue': 'Latin',
    'cumbia': 'Latin',
    'sierreño': 'Latin',
    'corrido': 'Latin',
    'vallenato': 'Latin',

    # REGGAE/CARIBBEAN
    'reggae': 'Reggae/Caribbean',
    'roots reggae': 'Reggae/Caribbean',
    'dub': 'Reggae/Caribbean',
    'lovers rock': 'Reggae/Caribbean',
    'ragga': 'Reggae/Caribbean',
    'reggae-pop': 'Reggae/Caribbean',
    'ska': 'Reggae/Caribbean',
    'rocksteady': 'Reggae/Caribbean',
    'third wave ska': 'Reggae/Caribbean',
    '2 tone': 'Reggae/Caribbean',
    'calypso': 'Reggae/Caribbean',
    'soca': 'Reggae/Caribbean',
    'dancehall': 'Reggae/Caribbean',
    'gospel reggae': 'Gospel/Christian/Religious',

    # CLASSICAL
    'classical': 'Classical',
    'contemporary classical': 'Classical',
    'modern classical': 'Classical',
    'cinematic classical': 'Classical',
    'post-minimalism': 'Classical',
    'opera': 'Classical',
    'orchestral': 'Classical',
    'operatic pop': 'Easy Listening/Vocal',
    'classical crossover': 'Classical',
    'baroque': 'Classical',
    'electroacoustic': 'Classical',

    # GOSPEL/CHRISTIAN/RELIGIOUS
    'gospel': 'Gospel/Christian/Religious',
    'contemporary gospel': 'Gospel/Christian/Religious',
    'southern gospel': 'Gospel/Christian/Religious',
    'christian rock': 'Gospel/Christian/Religious',
    'contemporary christian': 'Gospel/Christian/Religious',
    'christmas music': 'Gospel/Christian/Religious',
    'praise & worship': 'Gospel/Christian/Religious',

    # WORLD MUSIC
    'world fusion': 'World Music',
    'afrobeat': 'World Music',
    'afrobeats': 'World Music',
    'afro-funk': 'World Music',
    'soukous': 'World Music',
    'chalga': 'World Music',
    'maloya élektrik': 'World Music',
    'filmi': 'World Music',
    'persian pop': 'World Music',
    'amapiano': 'World Music',
    'néo-trad': 'World Music',
    'hindustani classical': 'World Music',
    'indian classical': 'World Music',
    'chanson française': 'World Music',
    'celtic': 'World Music',
    'klezmer': 'World Music',
    'fado': 'World Music',
    'flamenco': 'World Music',

    # EXPERIMENTAL/AVANT-GARDE
    'experimental': 'Experimental/Avant-Garde',
    'avant-garde': 'Experimental/Avant-Garde',
    'experimental rock': 'Experimental/Avant-Garde',
    'noise': 'Experimental/Avant-Garde',
    'harsh noise': 'Experimental/Avant-Garde',
    'black noise': 'Experimental/Avant-Garde',
    'sound art': 'Experimental/Avant-Garde',
    'sound collage': 'Experimental/Avant-Garde',
    'plunderphonics': 'Experimental/Avant-Garde',
    'epic collage': 'Experimental/Avant-Garde',
    'spamwave': 'Experimental/Avant-Garde',
    'dariacore': 'Experimental/Avant-Garde',
    'experimental electronic': 'Experimental/Avant-Garde',
    'art rock': 'Experimental/Avant-Garde',
    'math rock': 'Experimental/Avant-Garde',
    'avant-prog': 'Experimental/Avant-Garde',

    # EASY LISTENING/VOCAL
    'easy listening': 'Easy Listening/Vocal',
    'lounge': 'Easy Listening/Vocal',
    'cocktail nation': 'Easy Listening/Vocal',
    'comedy': 'Easy Listening/Vocal',
    'spoken word': 'Easy Listening/Vocal',
    "children's music": 'Easy Listening/Vocal',
    'instrumental': 'Easy Listening/Vocal',
    'musical': 'Easy Listening/Vocal',
    'music hall': 'Easy Listening/Vocal',
    'new age': 'Easy Listening/Vocal',
    'poetry': 'Easy Listening/Vocal',
    'standup comedy': 'Easy Listening/Vocal',
}

def categorize_genre_tags(tags):
    """Takes a list of genre tag strings (as stored in df_songs), returns sorted
    comma-separated string of major genre categories, or None if no match."""
    if tags is None or not isinstance(tags, list) or len(tags) == 0:
        return None
    categories = set()
    for genre in tags:
        key = genre.strip().lower()
        if key in musicbrainz_to_major_genre:
            categories.add(musicbrainz_to_major_genre[key])
    return ', '.join(sorted(categories)) if categories else None

def aggregate_categories(row):
    """Takes the union of song- and album-level major genre categories."""
    cats = set()
    for col in ['song_major_genre_category_musicbrainz', 'album_major_genre_category_musicbrainz']:
        val = row[col]
        if isinstance(val, str):
            for cat in val.split(', '):
                cats.add(cat.strip())
    return ', '.join(sorted(cats)) if cats else None

# Apply categorization
df_songs['song_major_genre_category_musicbrainz']   = df_songs['song_genre_tags_musicbrainz'].apply(categorize_genre_tags)
df_songs['album_major_genre_category_musicbrainz']  = df_songs['album_genre_tags_musicbrainz'].apply(categorize_genre_tags)
df_songs['aggregate_major_genre_category_musicbrainz'] = df_songs.apply(aggregate_categories, axis=1)

# Summary
for col, label in [
    ('song_major_genre_category_musicbrainz',      'Song-level'),
    ('album_major_genre_category_musicbrainz',     'Album-level'),
    ('aggregate_major_genre_category_musicbrainz', 'Aggregate'),
]:
    n = df_songs[col].notna().sum()
    print(f"{label:12}: {n:,} / {len(df_songs):,} rows categorized ({n/len(df_songs)*100:.1f}%)")

print(f"\nSample:")
cols = ['title', 'original_performer_name',
        'song_major_genre_category_musicbrainz',
        'album_major_genre_category_musicbrainz',
        'aggregate_major_genre_category_musicbrainz']
print(df_songs[df_songs['aggregate_major_genre_category_musicbrainz'].notna()][cols].sample(10).to_string())


Song-level  : 6,301 / 38,383 rows categorized (16.4%)
Album-level : 18,809 / 38,383 rows categorized (49.0%)
Aggregate   : 19,059 / 38,383 rows categorized (49.7%)

Sample:
                      title                        original_performer_name        song_major_genre_category_musicbrainz album_major_genre_category_musicbrainz   aggregate_major_genre_category_musicbrainz
38238           Verano Rosa                                 Karol G & Feid                                         None                                  Latin                                        Latin
4784     Long Lonely Nights                                            NaN                                         None                              Pop, Rock                                    Pop, Rock
23980     One Day At A Time        Tupac With Eminem Featuring The Outlawz                                         None                     Blues, Hip Hop/Rap                           Blues, Hip Hop/Rap
36590      

In [36]:
# Add genre count columns to df_songs
# Counts distinct major genre categories (comma-separated values) in:
#   - #_of_genres_song:      from song_major_genre_category_musicbrainz
#   - #_of_genres_aggregate: from aggregate_major_genre_category_musicbrainz
# NaN / uncategorized rows return 0

import pandas as pd

def count_genres(genre_category_string):
    """
    Count the number of distinct major genre categories.

    Examples:
        None → 0
        "Hip Hop/Rap" → 1
        "Country/Americana, Folk, Rock" → 3
    """
    if pd.isna(genre_category_string):
        return 0
    categories = [c.strip() for c in str(genre_category_string).split(',')]
    return len(categories)

df_songs['#_of_genres_song']      = df_songs['song_major_genre_category_musicbrainz'].apply(count_genres)
df_songs['#_of_genres_aggregate'] = df_songs['aggregate_major_genre_category_musicbrainz'].apply(count_genres)

print("Genre Count Summary:")
print("=" * 80)

print("\n#_of_genres_song:")
print(df_songs['#_of_genres_song'].value_counts().sort_index().to_string())
print(f"  Mean: {df_songs['#_of_genres_song'].mean():.2f}")

print("\n#_of_genres_aggregate:")
print(df_songs['#_of_genres_aggregate'].value_counts().sort_index().to_string())
print(f"  Mean: {df_songs['#_of_genres_aggregate'].mean():.2f}")

print("\n" + "=" * 80)
print("COMPLETE: Added '#_of_genres_song' and '#_of_genres_aggregate' to df_songs")
print("=" * 80)


Genre Count Summary:

#_of_genres_song:
#_of_genres_song
0    32082
1     3313
2     1748
3      841
4      298
5       74
6       20
7        6
8        1
  Mean: 0.29

#_of_genres_aggregate:
#_of_genres_aggregate
0     19324
1      7384
2      5155
3      3202
4      1605
5       792
6       380
7       204
8       134
9        69
10       40
11       25
12       10
13       12
14        8
15       22
16        8
17        9
  Mean: 1.17

COMPLETE: Added '#_of_genres_song' and '#_of_genres_aggregate' to df_songs


In [38]:
df_songs.sample(5
                )

,title,name,performer_pre_normalized,original_performer_name,original_performer_name_normalized,peak_pos,wks_on_chart,first_charting_year,musicbrainz_artist_id,musicbrainz_recording_mbid,...,sony,label_mbid,song_genre_tags_musicbrainz,song_tags_musicbrainz,album_genre_tags_musicbrainz,song_major_genre_category_musicbrainz,album_major_genre_category_musicbrainz,aggregate_major_genre_category_musicbrainz,#_of_genres_song,#_of_genres_aggregate
34405,Maybach,42 dugg,42 Dugg,42 Dugg Featuring Future,NaN,68,9,2021,4214aa04-64b0-442a-982b-0560e6d3a5bd,5f5fd3c9-c483-4c05-8e8a-dd72bce88861,...,False,"[420bafe0-10be-424c-afff-197b2a071d17, 592a3d1...",NaN,NaN,"[detroit trap, hip hop, trap]",None,Hip Hop/Rap,Hip Hop/Rap,0,1
26585,Stranded (Haiti Mon Amour),jay-z,Jay-Z,"Jay-Z, Bono, The Edge & Rihanna",NaN,16,2,2010,f82bcf78-5b69-4622-a5ef-73800768d9ac,05b1afc5-a2ae-4152-949c-eaa792374bea,...,False,[b2f6d997-568f-4c68-bc0d-050c899308a0],"[pop, hip hop]",NaN,"[contemporary country, contemporary r&b, count...","Hip Hop/Rap, Pop","Country/Americana, R&B/Soul/Funk","Country/Americana, Hip Hop/Rap, Pop, R&B/Soul/...",2,4
7284,Rainbow Ride,andy kim,Andy Kim,NaN,NaN,49,7,1968,a6d3a1bf-ee78-4a6d-96b3-1bd9769128c4,cde2da83-dc8e-4995-91af-9bf1f14fe64e,...,False,"[04d7bed5-4c4f-428a-a55d-101bd40034d2, 04d7bed...",NaN,NaN,"[pop rock, rock]",None,"Pop, Rock","Pop, Rock",0,2
10447,Helen Wheels,paul mccartney,Paul McCartney And Wings,NaN,paul mccartney and wings,10,13,1973,ba550d0e-adac-4864-b88b-407cab5e76af,22aaa808-2935-43ee-84d8-d56c7787a064,...,False,[98aebf66-50f9-4320-bb9b-8100bf4f03b9],NaN,NaN,"[pop, rock]",None,"Pop, Rock","Pop, Rock",0,2
16930,Le Bel Age (The Best Years),pat benatar,Pat Benatar,NaN,NaN,54,8,1986,6bc658ea-005f-486b-8f94-33a2e40f7a72,a44ad6d1-6993-49d6-872b-64936217bf62,...,False,[ed5601e5-7c54-426e-982a-1a208dd0b0ad],NaN,NaN,"[arena rock, blues, electronic, hard rock, new...",None,"Blues, Electronic/Dance, Pop, Rock","Blues, Electronic/Dance, Pop, Rock",0,4


In [40]:
# Deduplicate label and label_mbid arrays in df_songs
# Removes repeated entries caused by array_agg without DISTINCT in the original query
# Preserves order (first occurrence kept) and leaves NaN values unchanged

def dedup_list(val):
    if not isinstance(val, list):
        return val
    seen = []
    for item in val:
        if item not in seen:
            seen.append(item)
    return seen

df_songs['label']      = df_songs['label'].apply(dedup_list)
df_songs['label_mbid'] = df_songs['label_mbid'].apply(dedup_list)

# Verify
multi = df_songs['label'].apply(lambda x: isinstance(x, list) and len(x) > 1)
print(f"Rows with multiple labels after dedup: {multi.sum():,}")
print(df_songs[multi][['title', 'label', 'label_mbid']].head(5).to_string())


Rows with multiple labels after dedup: 4,423
                                title                                              label                                                                    label_mbid
84                          Pussy Cat            [Taragon Records, BMG Special Products]  [afdabfd9-9285-46a3-a911-5e0929fecd98, 7fc89221-5b85-43d6-aa72-0581aed43018]
144    Love Makes The World Go 'round                   [E.P.E Record, Enzo Productions]  [deaf74f6-0f81-4a42-9c07-b8cd5112a832, 0ace4bee-2f85-42d2-a8e1-002096b45bf1]
173  Come Closer To Me (Acercate Mas)  [Madacy Entertainment, EMI Music Special Markets]  [0afa4f14-70e2-479d-9b5e-9eaceeb25866, f6281082-8f72-437e-9814-942b1b30ccdd]
174                       Mr. Success                     [Universal Music, Music on CD]  [13a464dc-b9fd-4d16-a4f4-d4316f6a46c7, dd3ee43d-5a03-4cf4-9297-fe5ea557f40a]
176                 Come On, Let's Go         [Time–Life Music, Warner Special Products]  [3a657e07-7f99-4baf-ab3b-53cdc

In [41]:
# Save df_songs to CSV checkpoint
df_songs.to_csv(
    '/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/df_songs.csv',
    index=False
)

print(f"Saved df_songs: {len(df_songs):,} rows × {len(df_songs.columns)} columns")
print(f"\nColumns: {list(df_songs.columns)}")


Saved df_songs: 38,383 rows × 27 columns

Columns: ['title', 'name', 'performer_pre_normalized', 'original_performer_name', 'original_performer_name_normalized', 'peak_pos', 'wks_on_chart', 'first_charting_year', 'musicbrainz_artist_id', 'musicbrainz_recording_mbid', 'label', 'major_label', 'major_label_names', 'other_label', 'other_label_names', 'warner', 'universal', 'sony', 'label_mbid', 'song_genre_tags_musicbrainz', 'song_tags_musicbrainz', 'album_genre_tags_musicbrainz', 'song_major_genre_category_musicbrainz', 'album_major_genre_category_musicbrainz', 'aggregate_major_genre_category_musicbrainz', '#_of_genres_song', '#_of_genres_aggregate']
